In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash kailash-ml kailash-kaizen polars plotly gdown python-dotenv nest-asyncio

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP M5 inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated from shared/. Click the ▼ arrow on the left to hide.
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations


# ── kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model

# ── diagnostics.py ──
"""DL Diagnostics Toolkit — clinical instruments for deep-network training.

Wraps PyTorch forward/backward hooks into a student-friendly API that
surfaces the four failure modes professionals must recognise:

    1. Stethoscope  — loss-curve shape (under/over-fit, instability)
    2. Blood Test   — gradient flow per layer (vanishing / exploding)
    3. X-Ray        — activation statistics & dead neurons
    4. Prescription — automated diagnosis with actionable suggestions

Typical use inside an exercise::


    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()
        for epoch in range(epochs):
            for batch in dataloader:
                loss = train_step(model, batch)
                diag.record_batch(loss=loss.item(),
                                  lr=optimizer.param_groups[0]["lr"])
            diag.record_epoch(val_loss=evaluate(model, val_loader))
        diag.plot_training_dashboard().show()
        diag.report()

All DataFrames are Polars. All plots are Plotly. No matplotlib, no pandas.
"""

import logging
import math
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader


logger = logging.getLogger(__name__)

# Module names whose outputs are "dead-neuron sensitive" — we track fraction
# of zero outputs for these specifically.
_DEAD_NEURON_SENSITIVE: tuple[type[nn.Module], ...] = (
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
)

# Module names whose outputs we monitor for activation statistics.
_ACTIVATION_MONITORED: tuple[type[nn.Module], ...] = (
    nn.Linear,
    nn.Conv1d,
    nn.Conv2d,
    nn.Conv3d,
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
    nn.Tanh,
    nn.Sigmoid,
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.LayerNorm,
)


@dataclass
class _HookHandles:
    """Container for registered hook handles so we can detach cleanly."""

    gradient: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    activation: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    dead_neuron: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    grad_cam: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)

    def all(self) -> list[torch.utils.hooks.RemovableHandle]:
        return self.gradient + self.activation + self.dead_neuron + self.grad_cam


class DLDiagnostics:
    """Clinical instrumentation for PyTorch training loops.

    Collects per-batch time series of gradient norms, activation statistics,
    dead-neuron fractions, and losses; exposes Plotly visualisations and an
    automated report that surfaces overfitting, vanishing gradients, and
    pathological dead-ReLU layers.

    Args:
        model: The ``nn.Module`` to instrument. The model is NOT modified;
            only forward/backward hooks are attached.
        dead_neuron_threshold: Fraction of zero outputs above which a layer
            is flagged as "dead" in :meth:`report`. Defaults to ``0.5``.
        window: Number of recent batches used for dead-neuron statistics.
            Defaults to ``64``.

    Example:
        >>> model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
        >>> with DLDiagnostics(model) as diag:
        ...     diag.track_gradients()
        ...     diag.track_activations()
        ...     # ... training loop ...
    """

    def __init__(
        self,
        model: nn.Module,
        *,
        dead_neuron_threshold: float = 0.5,
        window: int = 64,
    ) -> None:
        if not isinstance(model, nn.Module):
            raise TypeError(
                f"DLDiagnostics requires an nn.Module; got {type(model).__name__}"
            )
        if not 0.0 < dead_neuron_threshold < 1.0:
            raise ValueError("dead_neuron_threshold must be in (0, 1)")
        if window < 1:
            raise ValueError("window must be >= 1")

        self.model = model
        self.device = get_device()
        self.dead_neuron_threshold = dead_neuron_threshold
        self.window = window

        # Time series storage — lists of dicts, converted to Polars on demand.
        self._grad_log: list[dict[str, Any]] = []
        self._act_log: list[dict[str, Any]] = []
        self._dead_log: list[dict[str, Any]] = []
        self._batch_log: list[dict[str, Any]] = []
        self._epoch_log: list[dict[str, Any]] = []

        # Running per-layer firing masks for dead-neuron detection.
        # Key: layer name -> tensor of firing counts per neuron (1D).
        self._firing_counts: dict[str, torch.Tensor] = {}
        self._firing_samples: dict[str, int] = {}

        # Counters bound to hook closures so they share scope.
        self._batch_idx = 0
        self._epoch_idx = 0

        self._handles = _HookHandles()
        self._tracking = {"gradients": False, "activations": False, "dead": False}

        # Grad-CAM capture buffers (populated on demand).
        self._gradcam_activation: torch.Tensor | None = None
        self._gradcam_gradient: torch.Tensor | None = None

        logger.info(
            "dldiagnostics.init",
            extra={
                "model_class": type(model).__name__,
                "device": str(self.device),
                "window": window,
            },
        )

    # ── Context-manager support ────────────────────────────────────────────

    def __enter__(self) -> DLDiagnostics:
        return self

    def __exit__(self, exc_type, exc, tb) -> None:  # noqa: D401
        self.detach()

    def __del__(self) -> None:  # pragma: no cover - best-effort cleanup
        try:
            self.detach()
        except Exception:
            # Finalizers MUST NOT raise. Silent cleanup is the documented
            # contract for __del__ in rules/patterns.md.
            pass

    # ── Hook registration ──────────────────────────────────────────────────

    def track_gradients(self) -> DLDiagnostics:
        """Register backward hooks on every trainable parameter.

        Records the L2 norm of each parameter's gradient at every backward
        pass, keyed by parameter name.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["gradients"]:
            return self
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            handle = param.register_hook(self._make_grad_hook(name))
            self._handles.gradient.append(handle)
        self._tracking["gradients"] = True
        logger.info(
            "dldiagnostics.track_gradients",
            extra={"hooks_registered": len(self._handles.gradient)},
        )
        return self

    def track_activations(self) -> DLDiagnostics:
        """Register forward hooks on monitored submodules.

        Records mean/std/min/max/dead-fraction of activations per layer
        at every forward pass.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["activations"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _ACTIVATION_MONITORED):
                continue
            handle = module.register_forward_hook(self._make_act_hook(name))
            self._handles.activation.append(handle)
        self._tracking["activations"] = True
        logger.info(
            "dldiagnostics.track_activations",
            extra={"hooks_registered": len(self._handles.activation)},
        )
        return self

    def track_dead_neurons(self) -> DLDiagnostics:
        """Register forward hooks on ReLU-family layers to track dead neurons.

        A "neuron" here is a channel (Conv) or output unit (Linear). The
        rolling fraction of batches where that neuron output zero is tracked.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["dead"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _DEAD_NEURON_SENSITIVE):
                continue
            handle = module.register_forward_hook(self._make_dead_hook(name))
            self._handles.dead_neuron.append(handle)
        self._tracking["dead"] = True
        logger.info(
            "dldiagnostics.track_dead_neurons",
            extra={"hooks_registered": len(self._handles.dead_neuron)},
        )
        return self

    def detach(self) -> None:
        """Remove ALL registered hooks and release references.

        Safe to call multiple times. Invoked automatically on context exit.
        """
        for handle in self._handles.all():
            try:
                handle.remove()
            except Exception:
                # Hook removal failures are benign (module may already be
                # gone). See rules/zero-tolerance.md Rule 3 carve-out for
                # cleanup paths.
                pass
        self._handles = _HookHandles()
        self._tracking = {k: False for k in self._tracking}
        self._gradcam_activation = None
        self._gradcam_gradient = None

    # ── Recording ─────────────────────────────────────────────────────────

    def record_batch(self, *, loss: float, lr: float | None = None) -> None:
        """Record per-batch scalar training signals.

        Args:
            loss: Training loss for the batch (post-backward).
            lr: Current learning rate (optional; read from optimizer).
        """
        if not math.isfinite(loss):
            logger.warning(
                "dldiagnostics.record_batch.nonfinite_loss",
                extra={"loss": str(loss), "batch": self._batch_idx},
            )
        self._batch_log.append(
            {
                "batch": self._batch_idx,
                "epoch": self._epoch_idx,
                "loss": float(loss),
                "lr": float(lr) if lr is not None else float("nan"),
            }
        )
        self._batch_idx += 1

    def record_epoch(
        self,
        *,
        val_loss: float | None = None,
        train_loss: float | None = None,
        **extra: float,
    ) -> None:
        """Record per-epoch summary metrics.

        Args:
            val_loss: Validation loss at epoch end.
            train_loss: Mean training loss for the epoch. If ``None``, it is
                computed from the batches in this epoch.
            **extra: Any additional scalar metrics to persist.
        """
        if train_loss is None:
            epoch_batches = [
                b for b in self._batch_log if b["epoch"] == self._epoch_idx
            ]
            if epoch_batches:
                train_loss = float(np.mean([b["loss"] for b in epoch_batches]))
        entry = {
            "epoch": self._epoch_idx,
            "train_loss": train_loss if train_loss is not None else float("nan"),
            "val_loss": val_loss if val_loss is not None else float("nan"),
            **{k: float(v) for k, v in extra.items()},
        }
        self._epoch_log.append(entry)
        self._epoch_idx += 1

    # ── Public DataFrame accessors ────────────────────────────────────────

    def gradients_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer gradient norms by batch."""
        if not self._grad_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "grad_norm": pl.Float64,
                    "grad_rms": pl.Float64,
                    "update_ratio": pl.Float64,
                }
            )
        return pl.DataFrame(self._grad_log)

    def activations_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer activation statistics by batch."""
        if not self._act_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "act_kind": pl.Utf8,
                    "mean": pl.Float64,
                    "std": pl.Float64,
                    "min": pl.Float64,
                    "max": pl.Float64,
                    "dead_fraction": pl.Float64,
                    "inactivity_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(self._act_log)

    def dead_neurons_df(self) -> pl.DataFrame:
        """Polars DataFrame of current per-layer dead-neuron fractions."""
        rows: list[dict[str, Any]] = []
        for name, counts in self._firing_counts.items():
            n_samples = max(self._firing_samples.get(name, 1), 1)
            dead_mask = (counts / n_samples) < 1e-6
            rows.append(
                {
                    "layer": name,
                    "n_neurons": int(counts.numel()),
                    "n_dead": int(dead_mask.sum().item()),
                    "dead_fraction": float(dead_mask.float().mean().item()),
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "layer": pl.Utf8,
                    "n_neurons": pl.Int64,
                    "n_dead": pl.Int64,
                    "dead_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def batches_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-batch scalars (loss, lr)."""
        if not self._batch_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "epoch": pl.Int64,
                    "loss": pl.Float64,
                    "lr": pl.Float64,
                }
            )
        return pl.DataFrame(self._batch_log)

    def epochs_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-epoch summary metrics."""
        if not self._epoch_log:
            return pl.DataFrame(
                schema={
                    "epoch": pl.Int64,
                    "train_loss": pl.Float64,
                    "val_loss": pl.Float64,
                }
            )
        return pl.DataFrame(self._epoch_log)

    # ── Plots ─────────────────────────────────────────────────────────────

    def plot_loss_curves(self) -> go.Figure:
        """Loss stethoscope: train vs val curves with overfitting callout.

        Returns:
            A Plotly Figure with two traces (train / val) and annotations
            for detected overfitting epoch (if any).
        """
        epochs = self.epochs_df()
        batches = self.batches_df()
        fig = go.Figure()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train (per batch)",
                    line=dict(color="lightblue", width=1),
                    opacity=0.5,
                )
            )
        if epochs.height:
            fig.add_trace(
                go.Scatter(
                    x=epochs["epoch"].to_list(),
                    y=epochs["train_loss"].to_list(),
                    mode="lines+markers",
                    name="train (epoch mean)",
                    line=dict(color="steelblue", width=2),
                )
            )
            if epochs["val_loss"].is_not_null().any():
                fig.add_trace(
                    go.Scatter(
                        x=epochs["epoch"].to_list(),
                        y=epochs["val_loss"].to_list(),
                        mode="lines+markers",
                        name="val",
                        line=dict(color="firebrick", width=2),
                    )
                )
        overfit_epoch = self._detect_overfit_epoch()
        if overfit_epoch is not None:
            fig.add_vline(
                x=overfit_epoch,
                line=dict(color="orange", dash="dash"),
                annotation_text=f"overfitting suspected @ epoch {overfit_epoch}",
                annotation_position="top",
            )
        fig.update_layout(
            title="Loss Curves (Stethoscope)",
            xaxis_title="step",
            yaxis_title="loss",
            template="plotly_white",
            hovermode="x unified",
        )
        return fig

    def plot_gradient_flow(self) -> go.Figure:
        """Blood test: per-layer gradient norm over time.

        One trace per tracked parameter. Layers whose gradient norm collapses
        toward zero are the vanishing-gradient culprits.
        """
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Gradient Flow (Blood Test) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["grad_norm"].to_list(),
                    mode="lines",
                    name=layer,
                    hovertemplate=f"{layer}<br>batch=%{{x}}<br>‖∇‖=%{{y:.3e}}",
                )
            )
        fig.update_layout(
            title="Gradient Flow by Layer (Blood Test)",
            xaxis_title="batch",
            yaxis_title="gradient L2 norm",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_activation_stats(self) -> go.Figure:
        """X-Ray: activation mean ± std per layer over time."""
        df = self.activations_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Activation Statistics (X-Ray) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["mean"].to_list(),
                    mode="lines",
                    name=f"{layer} — mean",
                    hovertemplate=(
                        f"{layer}<br>batch=%{{x}}<br>mean=%{{y:.3f}}<br>"
                        "std=%{customdata:.3f}"
                    ),
                    customdata=sub["std"].to_list(),
                )
            )
        fig.update_layout(
            title="Activation Mean per Layer (X-Ray)",
            xaxis_title="batch",
            yaxis_title="activation mean",
            template="plotly_white",
        )
        return fig

    def plot_dead_neurons(self) -> go.Figure:
        """X-Ray: fraction of dead neurons per ReLU-family layer."""
        df = self.dead_neurons_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Dead Neurons (X-Ray) — no ReLU-family layers tracked",
                template="plotly_white",
            )
            return fig
        colors = [
            "firebrick" if frac > self.dead_neuron_threshold else "steelblue"
            for frac in df["dead_fraction"].to_list()
        ]
        fig.add_trace(
            go.Bar(
                x=df["layer"].to_list(),
                y=df["dead_fraction"].to_list(),
                marker_color=colors,
                text=[
                    f"{int(n)}/{int(t)}" for n, t in zip(df["n_dead"], df["n_neurons"])
                ],
                textposition="outside",
            )
        )
        fig.add_hline(
            y=self.dead_neuron_threshold,
            line=dict(color="orange", dash="dash"),
            annotation_text=f"alert threshold ({self.dead_neuron_threshold:.0%})",
        )
        fig.update_layout(
            title="Dead Neuron Fraction by Layer (X-Ray)",
            xaxis_title="layer",
            yaxis_title="fraction dead",
            yaxis=dict(range=[0, 1]),
            template="plotly_white",
            showlegend=False,
        )
        return fig

    def plot_training_dashboard(self) -> go.Figure:
        """Vital signs: 2x2 dashboard (loss, grad flow, activations, LR)."""
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Loss (Stethoscope)",
                "Gradient Flow (Blood Test)",
                "Activation Mean (X-Ray)",
                "Learning Rate",
            ),
        )

        # (1,1) Loss
        epochs = self.epochs_df()
        batches = self.batches_df()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train loss",
                    line=dict(color="steelblue", width=1),
                ),
                row=1,
                col=1,
            )
        if epochs.height and epochs["val_loss"].is_not_null().any():
            # Place val loss at the last batch of each epoch for alignment.
            val_x = []
            for ep in epochs["epoch"].to_list():
                ep_batches = batches.filter(pl.col("epoch") == ep)
                val_x.append(
                    int(ep_batches["batch"].max()) if ep_batches.height else ep
                )
            fig.add_trace(
                go.Scatter(
                    x=val_x,
                    y=epochs["val_loss"].to_list(),
                    mode="lines+markers",
                    name="val loss",
                    line=dict(color="firebrick"),
                ),
                row=1,
                col=1,
            )

        # (1,2) Gradient flow
        gdf = self.gradients_df()
        if gdf.height:
            for layer in gdf["layer"].unique().to_list():
                sub = gdf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["grad_norm"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
            fig.update_yaxes(type="log", row=1, col=2)

        # (2,1) Activation mean
        adf = self.activations_df()
        if adf.height:
            for layer in adf["layer"].unique().to_list():
                sub = adf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["mean"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=2,
                    col=1,
                )

        # (2,2) LR
        if batches.height and batches["lr"].is_not_null().any():
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["lr"].to_list(),
                    mode="lines",
                    name="lr",
                    line=dict(color="darkgreen"),
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Training Dashboard (Vital Signs)",
            template="plotly_white",
            height=720,
        )
        return fig

    def plot_lr_vs_loss(self) -> go.Figure:
        """Plot LR vs loss (useful after an :meth:`lr_range_test`)."""
        df = self.batches_df()
        fig = go.Figure()
        if df.height == 0 or df["lr"].is_null().all():
            fig.update_layout(title="LR vs Loss — no data", template="plotly_white")
            return fig
        fig.add_trace(
            go.Scatter(
                x=df["lr"].to_list(),
                y=df["loss"].to_list(),
                mode="lines",
                line=dict(color="steelblue"),
            )
        )
        fig.update_layout(
            title="Learning Rate vs Loss",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_weight_distributions(self) -> go.Figure:
        """Histogram of weight values, one trace per parameter tensor."""
        fig = go.Figure()
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.ndim == 0:
                continue
            values = param.detach().cpu().flatten().numpy()
            fig.add_trace(go.Histogram(x=values, name=name, opacity=0.6))
        fig.update_layout(
            title="Weight Distributions",
            xaxis_title="value",
            yaxis_title="count",
            barmode="overlay",
            template="plotly_white",
        )
        return fig

    def plot_gradient_norms(self) -> go.Figure:
        """Mean gradient norm per layer across the run (bar chart)."""
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(title="Gradient Norms — no data", template="plotly_white")
            return fig
        summary = df.group_by("layer").agg(
            pl.col("grad_norm").mean().alias("mean_grad")
        )
        summary = summary.sort("mean_grad")
        fig.add_trace(
            go.Bar(
                x=summary["layer"].to_list(),
                y=summary["mean_grad"].to_list(),
                marker_color="steelblue",
            )
        )
        fig.update_layout(
            title="Mean Gradient Norm per Layer",
            xaxis_title="layer",
            yaxis_title="mean ‖∇‖",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    # ── Automated report ──────────────────────────────────────────────────

    def report(self) -> dict[str, Any]:
        """Print a human-readable diagnosis and return findings as a dict.

        Findings covered:
            * Gradient flow (vanishing / exploding / healthy)
            * Dead neurons (per-layer ReLU-family)
            * Loss trend (overfitting, underfitting, instability)

        Returns:
            A dict with keys ``gradient_flow``, ``dead_neurons``, ``loss_trend``
            each mapping to a dict with ``severity`` and ``message``.
        """
        findings: dict[str, Any] = {}

        # 1. Gradient flow — uses SCALE-INVARIANT per-element RMS (grad_rms)
        # and update_ratio (‖∇W‖/‖W‖), matching slide 5F thresholds.
        gdf = self.gradients_df()
        if gdf.height and "grad_rms" in gdf.columns:
            stats = gdf.group_by("layer").agg(
                pl.col("grad_rms").mean().alias("rms"),
                pl.col("update_ratio").mean().alias("ur"),
            )
            min_rms_raw = stats["rms"].min()
            max_rms_raw = stats["rms"].max()
            min_ur_raw = stats["ur"].min()
            max_ur_raw = stats["ur"].max()
            min_rms = (
                float(min_rms_raw) if isinstance(min_rms_raw, (int, float)) else None
            )
            max_rms = (
                float(max_rms_raw) if isinstance(max_rms_raw, (int, float)) else None
            )
            min_ur = float(min_ur_raw) if isinstance(min_ur_raw, (int, float)) else 0.0
            max_ur = float(max_ur_raw) if isinstance(max_ur_raw, (int, float)) else 0.0
            if min_rms is None or max_rms is None or min_rms == 0:
                severity = "UNKNOWN"
                message = "Insufficient gradient data."
            elif min_rms < 1e-5 or min_ur < 1e-4:
                # Vanishing: RMS < 1e-5 OR update ratio < 1e-4 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms").row(0, named=True)["layer"]
                    if min_rms < 1e-5
                    else stats.sort("ur").row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Vanishing gradients at '{worst_layer}' — "
                    f"min RMS = {min_rms:.2e}, min update_ratio = {min_ur:.2e}. "
                    "Fix: verify pre-norm layout (LayerNorm/RMSNorm before block), "
                    "add residual connections, switch to GELU/SiLU, or use Kaiming init."
                )
            elif max_rms > 1e-2 or max_ur > 0.1:
                # Exploding: RMS > 1e-2 OR update ratio > 0.1 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms", descending=True).row(0, named=True)["layer"]
                    if max_rms > 1e-2
                    else stats.sort("ur", descending=True).row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Exploding gradients at '{worst_layer}' — "
                    f"max RMS = {max_rms:.2e}, max update_ratio = {max_ur:.2e}. "
                    "Fix: add gradient clipping (or adaptive: ZClip/AGC), reduce LR, "
                    "verify initialization (Kaiming for ReLU, Xavier for Tanh)."
                )
            elif max_rms / max(min_rms, 1e-12) > 1e3:
                severity = "WARNING"
                message = (
                    f"Large RMS spread across layers (max/min = "
                    f"{max_rms / min_rms:.1e}). Deep layers may be learning unevenly."
                )
            else:
                severity = "HEALTHY"
                message = (
                    f"Gradient flow OK (RMS range {min_rms:.2e}–{max_rms:.2e}, "
                    f"update_ratio range {min_ur:.2e}–{max_ur:.2e})."
                )
            findings["gradient_flow"] = {"severity": severity, "message": message}
        elif gdf.height:
            # Legacy path for dataframes without the new columns.
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": (
                    "grad_rms field missing — re-run with the current library "
                    "version to get scale-invariant diagnosis."
                ),
            }
        else:
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": "No gradient tracking enabled — call track_gradients().",
            }

        # 2. Dead neurons / saturation — uses ACTIVATION-TYPE-AWARE
        # inactivity_fraction from _act_log (ReLU: |x|<eps; Tanh: |x|>0.99;
        # Sigmoid: |x|>0.99 or |x|<0.01). This catches near-dead ReLU channels
        # that the legacy exact-zero check misses post-BatchNorm.
        adf = self.activations_df()
        if adf.height and "inactivity_fraction" in adf.columns:
            # Aggregate mean inactivity per layer (averaged across batches).
            agg = (
                adf.filter(pl.col("act_kind") != "other")
                .group_by("layer")
                .agg(
                    pl.col("inactivity_fraction").mean().alias("mean_inactive"),
                    pl.col("act_kind").first().alias("kind"),
                )
                .sort("mean_inactive", descending=True)
            )
            if agg.height:
                worst = agg.row(0, named=True)
                thr = self.dead_neuron_threshold
                if worst["mean_inactive"] > thr:
                    kind = worst["kind"]
                    if kind == "relu":
                        label = "dead neurons"
                        fix = "Switch to GELU/LeakyReLU or re-initialise with Kaiming."
                    elif kind == "tanh":
                        label = "saturated (|x|>0.99)"
                        fix = "Reduce LR, add LayerNorm before, or switch to GELU."
                    elif kind == "sigmoid":
                        label = "saturated (|x|>0.99 or |x|<0.01)"
                        fix = "Reduce LR, add BN/LN, or switch to softmax+CE if classification."
                    else:
                        label = "inactive"
                        fix = "Investigate activation distribution."
                    findings["dead_neurons"] = {
                        "severity": "WARNING",
                        "message": (
                            f"'{worst['layer']}' ({kind}): "
                            f"{worst['mean_inactive']:.0%} {label}. {fix}"
                        ),
                    }
                else:
                    findings["dead_neurons"] = {
                        "severity": "HEALTHY",
                        "message": (
                            f"All {agg.height} activation layers healthy "
                            f"(worst: {worst['layer']} at "
                            f"{worst['mean_inactive']:.0%} inactive, below "
                            f"{thr:.0%} threshold)."
                        ),
                    }
            else:
                findings["dead_neurons"] = {
                    "severity": "UNKNOWN",
                    "message": "No activation layers tracked — call track_activations().",
                }
        else:
            findings["dead_neurons"] = {
                "severity": "UNKNOWN",
                "message": "No activation tracking enabled — call track_activations().",
            }

        # 3. Loss trend
        edf = self.epochs_df()
        if edf.height >= 3 and edf["val_loss"].is_not_null().any():
            overfit = self._detect_overfit_epoch()
            train_trend = self._slope(edf["train_loss"].to_list())
            if overfit is not None:
                severity = "WARNING"
                message = (
                    f"Overfitting suspected at epoch {overfit} "
                    "(val loss rising while train loss falls). "
                    "Consider dropout, weight decay, or early stopping."
                )
            elif train_trend > -1e-4 and edf.height >= 5:
                severity = "WARNING"
                message = (
                    f"Underfitting — train loss slope {train_trend:.2e}/epoch. "
                    "Consider a larger model, more epochs, or higher LR."
                )
            else:
                severity = "HEALTHY"
                message = f"Loss converging (train slope {train_trend:.2e}/epoch)."
            findings["loss_trend"] = {"severity": severity, "message": message}
        else:
            findings["loss_trend"] = {
                "severity": "UNKNOWN",
                "message": "Need at least 3 epochs with val_loss for trend analysis.",
            }

        # Human-readable summary
        print("\n" + "═" * 64)
        print("  DL Diagnostics Report — Prescription Pad")
        print("═" * 64)
        icons = {
            "HEALTHY": "✓",
            "WARNING": "!",
            "CRITICAL": "X",
            "UNKNOWN": "?",
        }
        titles = {
            "gradient_flow": "Gradient flow",
            "dead_neurons": "Dead neurons ",
            "loss_trend": "Loss trend   ",
        }
        for key, label in titles.items():
            f = findings[key]
            print(
                f"  [{icons[f['severity']]}] {label} ({f['severity']}): {f['message']}"
            )
        print("═" * 64 + "\n")
        return findings

    # ── Grad-CAM ──────────────────────────────────────────────────────────

    def grad_cam(
        self,
        input_tensor: torch.Tensor,
        target_class: int,
        layer_name: str,
    ) -> torch.Tensor:
        """Compute a Grad-CAM heatmap for ``target_class`` from ``layer_name``.

        Args:
            input_tensor: Input batch ``(N, C, H, W)`` or ``(N, C, L)``.
            target_class: Target class index to explain.
            layer_name: ``model.named_modules()`` key of the conv layer to
                attribute. Usually the last convolutional layer.

        Returns:
            Heatmap tensor with shape matching the spatial dims of the tracked
            layer's output (``(N, H', W')`` for 2D inputs).

        Raises:
            ValueError: If ``layer_name`` is not found in the model.
        """
        target_module: nn.Module | None = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                target_module = module
                break
        if target_module is None:
            raise ValueError(
                f"Layer '{layer_name}' not found in model. "
                f"Available: {[n for n, _ in self.model.named_modules() if n][:10]}..."
            )

        self._gradcam_activation = None
        self._gradcam_gradient = None

        def fwd_hook(_mod: nn.Module, _inp: Any, out: torch.Tensor) -> None:
            self._gradcam_activation = out.detach()

        def bwd_hook(_mod: nn.Module, _gi: Any, go: Any) -> None:
            # go is a tuple — first element is d(output)/d(module-output)
            self._gradcam_gradient = go[0].detach()

        h1 = target_module.register_forward_hook(fwd_hook)
        h2 = target_module.register_full_backward_hook(bwd_hook)
        self._handles.grad_cam.extend([h1, h2])

        # Preserve caller's train/eval state — critical for mid-training use.
        was_training = self.model.training

        try:
            self.model.eval()
            inp = input_tensor.to(self.device)
            logits = self.model(inp)
            if logits.ndim != 2:
                raise ValueError(
                    f"grad_cam expects classification logits (N, C); got {logits.shape}"
                )
            self.model.zero_grad(set_to_none=True)
            one_hot = torch.zeros_like(logits)
            one_hot[:, target_class] = 1.0
            logits.backward(gradient=one_hot, retain_graph=False)

            if self._gradcam_activation is None or self._gradcam_gradient is None:
                raise RuntimeError(
                    "Grad-CAM hooks did not fire — layer may be unreachable "
                    "from the forward path."
                )
            activations = self._gradcam_activation  # (N, K, ...)
            gradients = self._gradcam_gradient  # (N, K, ...)
            # Global-average-pool gradients over spatial dims to get per-channel weights.
            spatial_dims = tuple(range(2, gradients.ndim))
            weights = gradients.mean(dim=spatial_dims, keepdim=True)  # (N, K, 1, ...)
            cam = (weights * activations).sum(dim=1)  # (N, ...)
            cam = torch.relu(cam)
            # Normalise per-sample to [0, 1].
            flat = cam.flatten(start_dim=1)
            mins = flat.min(dim=1, keepdim=True).values
            maxs = flat.max(dim=1, keepdim=True).values
            denom = (maxs - mins).clamp(min=1e-8)
            flat = (flat - mins) / denom
            cam = flat.view_as(cam)
            return cam.cpu()
        finally:
            # Restore caller's train/eval state BEFORE removing hooks, so
            # downstream errors in hook cleanup don't leave model in eval mode.
            if was_training:
                self.model.train()
            h1.remove()
            h2.remove()
            # Remove from bookkeeping too so detach() stays idempotent.
            self._handles.grad_cam = [
                h for h in self._handles.grad_cam if h is not h1 and h is not h2
            ]
            self._gradcam_activation = None
            self._gradcam_gradient = None

    # ── LR range test (static) ────────────────────────────────────────────

    @staticmethod
    def lr_range_test(
        model: nn.Module,
        dataloader: DataLoader,
        *,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD,
        lr_min: float = 1e-7,
        lr_max: float = 10.0,
        steps: int = 200,
        device: torch.device | None = None,
        batch_adapter: Callable[[Any], tuple[torch.Tensor, torch.Tensor]] | None = None,
        safety_divisor: float = 10.0,
    ) -> dict[str, Any]:
        """Leslie Smith LR range test (with fastai-style safe-LR recipe).

        Trains for ``steps`` batches while exponentially increasing LR from
        ``lr_min`` to ``lr_max``. Smooths the loss curve with EMA (beta=0.98)
        before finding the steepest-descent point — this matches fastai's
        ``lr_find()`` and avoids picking a single lucky batch in the tail.

        **Critical**: returns BOTH ``min_loss_lr`` (steepest descent) AND
        ``safe_lr = min_loss_lr / safety_divisor`` (default 10). Use ``safe_lr``
        in your optimizer — ``min_loss_lr`` is the edge of instability.

        The model's weights are saved before the test and **restored** on exit
        (via state_dict deepcopy), so calling this does not corrupt your model.

        Args:
            model: The model to probe. Weights are restored after return.
            dataloader: Any DataLoader. By default yields ``(inputs, targets)``
                tuples; pass ``batch_adapter`` for HF/SSL batch formats.
            loss_fn: Loss function (REQUIRED — no silent default).
            optimizer_cls: Optimizer class (default SGD).
            lr_min, lr_max, steps: Sweep configuration.
            device: Override compute device (default: model's current device).
            batch_adapter: Callable ``batch -> (x, y)``. Default handles
                tuple/list; pass a lambda for dict-shaped batches (e.g. HF).
            safety_divisor: Divide steepest-descent LR by this to get safe LR
                (fastai default: 10).

        Returns:
            ``{"safe_lr": float,              # use this in your optimizer
                "min_loss_lr": float,          # steepest descent (edge of instability)
                "divergence_lr": float,        # where smoothed loss > 4× min
                "lrs": [...], "losses": [...], "losses_smooth": [...],
                "figure": go.Figure}``
        """
        if steps < 2:
            raise ValueError("steps must be >= 2")
        if not 0 < lr_min < lr_max:
            raise ValueError("require 0 < lr_min < lr_max")
        if loss_fn is None:
            raise ValueError(
                "loss_fn is required — no silent default. "
                "Pass nn.CrossEntropyLoss() for classification or nn.MSELoss() for regression."
            )

        import copy as _copy

        # Device: honor model's current device unless overridden.
        dev = device or next(model.parameters()).device
        if device is not None:
            model = model.to(dev)

        # Save weights for restoration.
        saved_state = _copy.deepcopy(model.state_dict())

        # Default batch adapter handles tuple/list; raises loudly on dicts.
        def _default_adapter(batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
            if isinstance(batch, (tuple, list)) and len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(
                "lr_range_test got a non-(x,y) batch. Pass batch_adapter=... "
                "for HuggingFace-style dict batches or SSL single-tensor batches."
            )

        adapter = batch_adapter or _default_adapter

        try:
            optimizer = optimizer_cls(model.parameters(), lr=lr_min)
            gamma = (lr_max / lr_min) ** (1.0 / (steps - 1))
            lrs: list[float] = []
            losses: list[float] = []
            running_min = float("inf")  # O(1) running min, not O(n)
            model.train()
            data_iter = iter(dataloader)
            for step in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                x, y = adapter(batch)
                x, y = x.to(dev), y.to(dev)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr_min * (gamma**step)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                cur_lr = optimizer.param_groups[0]["lr"]
                cur_loss = float(loss.item())
                lrs.append(cur_lr)
                losses.append(cur_loss)
                if cur_loss < running_min:
                    running_min = cur_loss
                if not math.isfinite(cur_loss) or cur_loss > 10 * running_min:
                    logger.info(
                        "dldiagnostics.lr_range_test.diverged",
                        extra={"step": step, "lr": cur_lr, "loss": cur_loss},
                    )
                    break
        finally:
            # Always restore weights — no silent corruption.
            model.load_state_dict(saved_state)

        # EMA-smoothed losses (fastai convention, beta=0.98).
        beta = 0.98
        losses_smooth: list[float] = []
        ema = 0.0
        for i, ell in enumerate(losses):
            ema = beta * ema + (1 - beta) * ell
            # Bias-correct the EMA.
            losses_smooth.append(ema / (1 - beta ** (i + 1)))

        # min_loss_lr = LR at the steepest-descent point of SMOOTHED loss.
        min_loss_lr = lr_min
        divergence_lr = lr_max
        if len(losses_smooth) >= 3:
            dl = np.diff(np.array(losses_smooth))
            idx = int(np.argmin(dl))
            min_loss_lr = lrs[idx]
            # Divergence = first point where smoothed loss > 4× its minimum.
            min_smooth = min(losses_smooth)
            for i, s in enumerate(losses_smooth):
                if s > 4 * min_smooth and i > idx:
                    divergence_lr = lrs[i]
                    break

        safe_lr = min_loss_lr / safety_divisor

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses,
                mode="lines",
                name="raw loss",
                line=dict(color="lightgray"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses_smooth,
                mode="lines",
                name="smoothed",
                line=dict(color="#0D9488", width=2),
            )
        )
        fig.add_vline(
            x=safe_lr,
            line=dict(color="#22C55E", dash="dash"),
            annotation_text=f"safe_lr = {safe_lr:.2e}",
        )
        fig.add_vline(
            x=min_loss_lr,
            line=dict(color="#F59E0B", dash="dot"),
            annotation_text=f"min_loss_lr = {min_loss_lr:.2e}",
        )
        fig.update_layout(
            title="LR Range Test (smoothed) — use safe_lr, not min_loss_lr",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        logger.info(
            "dldiagnostics.lr_range_test.ok",
            extra={
                "safe_lr": safe_lr,
                "min_loss_lr": min_loss_lr,
                "divergence_lr": divergence_lr,
                "steps_run": len(lrs),
            },
        )
        return {
            "safe_lr": safe_lr,
            "min_loss_lr": min_loss_lr,
            "divergence_lr": divergence_lr,
            "suggested_lr": safe_lr,  # backwards-compat alias
            "lrs": lrs,
            "losses": losses,
            "losses_smooth": losses_smooth,
            "figure": fig,
        }

    # ── Hook factories (internal) ─────────────────────────────────────────

    def _make_grad_hook(self, name: str):
        """Scale-invariant gradient tracking.

        Records three metrics per layer per batch:
          - grad_norm: raw L2 norm (preserved for backwards compatibility)
          - grad_rms: per-element RMS = ‖∇‖ / sqrt(numel) — scale-invariant,
            comparable across layers of any size. This is what LLM training
            dashboards (W&B, TensorBoard) actually display.
          - update_ratio: ‖∇W‖ / ‖W‖ — the "effective LR" per layer.

        Casts to fp32 before reduction so BF16/FP16 gradients don't silently
        produce inf/NaN.
        """
        # Look up the parameter tensor for update-ratio computation.
        try:
            param = dict(self.model.named_parameters())[name]
        except KeyError:
            param = None

        def _hook(grad: torch.Tensor) -> None:
            try:
                # Handle sparse gradients (e.g. nn.Embedding(sparse=True)).
                g = grad.coalesce().values() if grad.is_sparse else grad
                # Cast to fp32 to avoid BF16/FP16 inf.
                g_f32 = g.detach().float()
                l2 = float(g_f32.norm(p=2).item())
                numel = max(g_f32.numel(), 1)
                rms = l2 / (numel**0.5)
                # Update ratio — use detached param weight norm.
                if param is not None:
                    p_norm = float(param.detach().float().norm(p=2).item())
                    upd_ratio = l2 / max(p_norm, 1e-12)
                else:
                    upd_ratio = 0.0
            except Exception as exc:  # pragma: no cover - defensive
                logger.warning(
                    "dldiagnostics.grad_hook.error",
                    extra={"layer": name, "error": str(exc)},
                )
                return
            self._grad_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "grad_norm": l2,  # preserved for compatibility
                    "grad_rms": rms,  # scale-invariant
                    "update_ratio": upd_ratio,  # ‖∇W‖ / ‖W‖
                }
            )

        return _hook

    def _make_act_hook(self, name: str):
        """Activation statistics. Casts to fp32 to avoid BF16/FP16 overflow.

        The `inactivity_fraction` field measures activation-type-appropriate
        pathology:
          - ReLU / GELU / SiLU: fraction with |x| < 1e-6 (dead neurons)
          - Tanh: fraction with |x| > 0.99 (saturated)
          - Sigmoid: fraction with |x| > 0.99 OR |x| < 0.01 (saturated)
          - Others: 0 (no pathology definition)

        The legacy `dead_fraction` field (exact-zero count) is preserved for
        backwards compatibility but is only meaningful for ReLU.
        """
        # Determine activation type from module class name for dispatching.
        act_kind = self._classify_activation_module(name)

        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            # Cast to fp32 for numerical safety (BF16/FP16 overflow on .std()).
            out = output.detach().float()
            try:
                mean = float(out.mean().item())
                std = float(out.std().item()) if out.numel() > 1 else 0.0
                mn = float(out.min().item())
                mx = float(out.max().item())
                legacy_dead = float((out == 0).float().mean().item())
                # Activation-aware inactivity/saturation metric.
                if act_kind == "relu":
                    inactivity = float((out.abs() < 1e-6).float().mean().item())
                elif act_kind == "tanh":
                    inactivity = float((out.abs() > 0.99).float().mean().item())
                elif act_kind == "sigmoid":
                    inactivity = float(
                        ((out > 0.99) | (out < 0.01)).float().mean().item()
                    )
                else:
                    inactivity = 0.0
            except RuntimeError:
                # Non-numeric tensor (e.g. mixed dtype). Skip silently.
                return
            # Guard against NaN/inf leaking into logs.
            for val_name, val in (("mean", mean), ("std", std)):
                if val != val or val in (float("inf"), float("-inf")):
                    logger.warning(
                        "dldiagnostics.act_hook.nonfinite",
                        extra={"layer": name, "field": val_name},
                    )
                    return
            self._act_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "act_kind": act_kind,
                    "mean": mean,
                    "std": std,
                    "min": mn,
                    "max": mx,
                    "dead_fraction": legacy_dead,
                    "inactivity_fraction": inactivity,
                }
            )

        return _hook

    def _classify_activation_module(self, name: str) -> str:
        """Return one of 'relu', 'tanh', 'sigmoid', 'other' for a module name."""
        try:
            mod = dict(self.model.named_modules())[name]
        except KeyError:
            return "other"
        cls = type(mod).__name__.lower()
        if any(k in cls for k in ("relu", "gelu", "silu", "swish", "elu")):
            return "relu"
        if "tanh" in cls:
            return "tanh"
        if "sigmoid" in cls:
            return "sigmoid"
        return "other"

    def _make_dead_hook(self, name: str):
        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            out = output.detach()
            # Collapse all non-channel dims. Convention: channel dim = 1 for
            # Conv outputs (N, C, ...); for Linear, output is (N, F) — here
            # we treat dim 1 as "neurons" which matches both shapes.
            if out.ndim < 2:
                return
            reduce_dims = tuple(d for d in range(out.ndim) if d != 1)
            fired = (out != 0).any(dim=reduce_dims).float().cpu()
            if name not in self._firing_counts:
                self._firing_counts[name] = torch.zeros_like(fired)
                self._firing_samples[name] = 0
            self._firing_counts[name] += fired
            self._firing_samples[name] += 1
            # Keep memory bounded by decaying old counts when window exceeded.
            if self._firing_samples[name] > self.window:
                scale = self.window / self._firing_samples[name]
                self._firing_counts[name] = self._firing_counts[name] * scale
                self._firing_samples[name] = self.window

        return _hook

    # ── Internal analysis helpers ─────────────────────────────────────────

    @staticmethod
    def _slope(series: list[float]) -> float:
        """Least-squares slope of a 1D series vs its index (ignores NaN)."""
        xs = np.arange(len(series), dtype=float)
        ys = np.asarray(series, dtype=float)
        mask = np.isfinite(ys)
        if mask.sum() < 2:
            return 0.0
        xs, ys = xs[mask], ys[mask]
        return float(np.polyfit(xs, ys, 1)[0])

    def _detect_overfit_epoch(self) -> int | None:
        """Return epoch index where val loss starts rising while train falls."""
        edf = self.epochs_df()
        if edf.height < 3:
            return None
        train = edf["train_loss"].to_list()
        val = edf["val_loss"].to_list()
        for i in range(2, len(val)):
            v_now, v_prev = val[i], val[i - 1]
            t_now, t_prev = train[i], train[i - 1]
            if any(
                x is None or not math.isfinite(x)
                for x in (v_now, v_prev, t_now, t_prev)
            ):
                continue
            if v_now > v_prev and t_now < t_prev:
                # Confirm trend persists one step back if available.
                return i
        return None


# ════════════════════════════════════════════════════════════════════════
# Standalone diagnostic checkpoint — for use AFTER a training loop
# ════════════════════════════════════════════════════════════════════════


def run_diagnostic_checkpoint(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: Callable[..., Any],
    *,
    title: str = "Model",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    batch_adapter: Callable[[Any], tuple[torch.Tensor, ...]] | None = None,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Run a short instrumented diagnostic pass on a TRAINED model.

    Use this between the Train and Visualise phases of an exercise. It
    attaches the four diagnostic instruments (gradients, activations,
    dead-neurons, scalar history) to the model, runs ``n_batches`` of
    forward-backward passes to populate the history, replays any
    epoch-level losses you collected during real training, and prints the
    Prescription Pad with auto-diagnosis.

    The model's weights are NOT updated — gradients are computed but no
    optimizer step is taken. The model's train/eval state is preserved.

    Args:
        model: A trained ``nn.Module``.
        dataloader: Any DataLoader whose batches the loss function accepts.
        loss_fn: Callable. The default contract is
            ``loss_fn(model, batch) -> (scalar_loss, extras_dict)`` matching
            the M5 exercise convention. Pass ``batch_adapter`` to wrap a
            different signature.
        title: Human-readable name printed in the dashboard title.
        n_batches: How many batches to instrument (default 8 — enough for
            a stable mean per layer without slowing the exercise down).
        train_losses: Optional list of per-epoch train losses captured
            during the actual training run. Replayed into the dashboard's
            stethoscope view so students see the real loss trajectory, not
            just the diagnostic-pass losses.
        val_losses: Optional list of per-epoch validation losses, same
            length as ``train_losses``.
        show: If ``True``, calls ``.show()`` on the dashboard figure.
        batch_adapter: Optional ``batch -> (loss_fn_args...)`` translator
            for non-standard batch shapes.

    Returns:
        ``(diag, findings)`` so the caller can inspect the DataFrames and
        the Prescription Pad findings dict.
    """
    if not isinstance(model, nn.Module):
        raise TypeError("run_diagnostic_checkpoint requires an nn.Module")
    if n_batches < 1:
        raise ValueError("n_batches must be >= 1")

    diag = DLDiagnostics(model)
    diag.track_gradients().track_activations().track_dead_neurons()

    # Replay real training history into the dashboard if provided.
    if train_losses or val_losses:
        n_epochs = max(len(train_losses or []), len(val_losses or []))
        for i in range(n_epochs):
            tl = (
                float(train_losses[i])
                if train_losses and i < len(train_losses)
                else None
            )
            vl = float(val_losses[i]) if val_losses and i < len(val_losses) else None
            # Synthesise one batch entry per epoch so the per-batch stethoscope
            # has data to plot — students still see the real epoch losses.
            if tl is not None:
                diag.record_batch(loss=tl)
            diag.record_epoch(train_loss=tl, val_loss=vl)

    was_training = model.training
    try:
        model.train()  # Enable gradients + activation hooks.
        seen = 0
        for batch in dataloader:
            if seen >= n_batches:
                break
            try:
                if batch_adapter is not None:
                    args = batch_adapter(batch)
                    loss_out = loss_fn(model, *args)
                else:
                    loss_out = loss_fn(model, batch)
                # Convention: M5 loss_fn returns (loss, extras_dict). Allow
                # either a bare tensor or a tuple.
                if isinstance(loss_out, tuple):
                    loss = loss_out[0]
                else:
                    loss = loss_out
                if not isinstance(loss, torch.Tensor):
                    raise TypeError(
                        f"loss_fn returned {type(loss).__name__}; expected Tensor"
                    )
                model.zero_grad(set_to_none=True)
                loss.backward()
                # NOTE: no optimizer.step() — diagnostic pass is read-only.
                diag.record_batch(loss=float(loss.item()))
            except Exception as exc:  # pragma: no cover - student loop variability
                logger.warning(
                    "dldiagnostics.checkpoint.batch_skipped",
                    extra={"batch_idx": seen, "error": str(exc)},
                )
            seen += 1
    finally:
        if not was_training:
            model.eval()

    # Render the dashboard and the Prescription Pad.
    fig = diag.plot_training_dashboard()
    fig.update_layout(title=f"{title} — Diagnostic Dashboard (Vital Signs)")
    if show:
        try:
            fig.show()
        except Exception as exc:  # pragma: no cover - no display in CI
            logger.info(
                "dldiagnostics.checkpoint.show_skipped",
                extra={"error": str(exc)},
            )

    findings = diag.report()
    return diag, findings


def diagnose_classifier(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Classifier",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` cross-entropy classifiers.

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.cross_entropy(model(x), y)``. Use this for
    CNN, transformer, and transfer-learning exercises.

    Args:
        model: Trained classifier ``nn.Module``.
        dataloader: Yields ``(x, y)`` tuples; ``y`` is class indices.
        title: Title for the dashboard.
        n_batches: Batches to instrument.
        train_losses, val_losses: Optional epoch histories to replay.
        show: Show the figure inline.
        forward_returns_tuple: If ``True``, ``model(x)`` returns
            ``(logits, ...)`` (e.g. attention models) — only the first
            element is used as logits.

    Returns:
        ``(diag, findings)``
    """
    import torch.nn.functional as F  # local — torch already imported

    def _cls_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        logits = out[0] if forward_returns_tuple else out
        return F.cross_entropy(logits, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _cls_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


def diagnose_regressor(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Regressor",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` MSE regressors (RNN exercises).

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.mse_loss(model(x), y)``.

    Args:
        forward_returns_tuple: Set ``True`` for attention models that
            return ``(predictions, attn_weights)``.
    """
    import torch.nn.functional as F

    def _reg_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        pred = out[0] if forward_returns_tuple else out
        return F.mse_loss(pred, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _reg_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


__all__ = [
    "DLDiagnostics",
    "run_diagnostic_checkpoint",
    "diagnose_classifier",
    "diagnose_regressor",
]

# ── ex_5.py ──
"""
Shared utilities for Exercise 5 — GANs and Generative Models.

Infrastructure only: data loading, visualisation helpers, metric computation,
and kailash-ml engine setup. No domain logic or business scenarios.
"""

import asyncio
import pickle
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import matplotlib.pyplot as plt

from kailash.db import ConnectionManager
from kailash_ml import ModelVisualizer
from kailash_ml.engines.experiment_tracker import ExperimentTracker
from kailash_ml.engines.model_registry import ModelRegistry
from kailash_ml.types import MetricSpec


if TYPE_CHECKING:
    from kailash_ml.engines.model_registry import ModelVersion

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
LATENT_DIM = 64
IMG_DIM = 28 * 28
BATCH_SIZE = 128
REPO_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "mnist"
OUTPUT_DIR = Path(__file__).resolve().parent


# ════════════════════════════════════════════════════════════════════════
# Environment and device
# ════════════════════════════════════════════════════════════════════════
def init_environment() -> torch.device:
    """Set up environment, seeds, and return the compute device."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ════════════════════════════════════════════════════════════════════════
# Data loading
# ════════════════════════════════════════════════════════════════════════
def load_mnist(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, DataLoader]:
    """Load full MNIST (60K), scale to [-1, 1] for tanh generators.

    Returns:
        X_real: (60000, 1, 28, 28) tensor on device, range [-1, 1]
        y_real: (60000,) long tensor on device
        real_loader: DataLoader with batch_size=128, shuffle, drop_last
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_real = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_real = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))], dtype=torch.long
    )
    X_real = (X_real * 2.0 - 1.0).to(device)
    y_real = y_real.to(device)

    print(
        f"MNIST: {len(X_real)} digits, shape {tuple(X_real.shape[1:])}, "
        f"pixel range [{X_real.min():.2f}, {X_real.max():.2f}]"
    )
    class_dist = ", ".join(f"{c}={int((y_real == c).sum())}" for c in range(10))
    print(f"  class distribution: {class_dist}")

    real_loader = DataLoader(
        TensorDataset(X_real), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    return X_real, y_real, real_loader


# ════════════════════════════════════════════════════════════════════════
# Kailash engine setup
# ════════════════════════════════════════════════════════════════════════
def setup_engines() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None]
):
    """Create ExperimentTracker and ModelRegistry for GAN experiments.

    Returns:
        conn, tracker, experiment_name, registry (or None)
    """

    async def _setup():
        conn = ConnectionManager("sqlite:///mlfp05_gans.db")
        await conn.initialize()
        tracker = ExperimentTracker(conn)
        exp_name = await tracker.create_experiment(
            name="m5_gans",
            description="GAN variants on MNIST (60K images)",
        )
        try:
            registry = ModelRegistry(conn)
        except Exception as e:
            registry = None
            print(f"  Note: ModelRegistry setup skipped ({e})")
        return conn, tracker, exp_name, registry

    return asyncio.run(_setup())


async def close_engines(conn: ConnectionManager) -> None:
    """Cleanly shut down the connection manager."""
    await conn.close()


# ════════════════════════════════════════════════════════════════════════
# Generator and Discriminator architectures
# ════════════════════════════════════════════════════════════════════════
class Generator(nn.Module):
    """MLP Generator: z -> 784-d -> (1, 28, 28).

    Uses BatchNorm + LeakyReLU (DCGAN best practices) and Tanh output
    to match the [-1, 1] image scaling.
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    """MLP Discriminator: 28x28 -> scalar logit.

    Dropout prevents D from overfitting to real images (memorising
    instead of learning distributional features).
    """

    def __init__(self, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_DIM, hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# FID feature extractor
# ════════════════════════════════════════════════════════════════════════
class LeNetFeatureExtractor(nn.Module):
    """CNN feature extractor for FID computation.

    Returns 64-dim feature vectors (analogous to InceptionV3 pool3 layer).
    We use a domain-specific extractor because InceptionV3 expects 299x299 RGB;
    for 28x28 grayscale MNIST a trained LeNet gives more meaningful distances.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def train_feature_extractor(
    X_real: torch.Tensor,
    y_real: torch.Tensor,
    device: torch.device,
    epochs: int = 5,
) -> LeNetFeatureExtractor:
    """Train the LeNet feature extractor on MNIST for FID computation.

    Returns the trained extractor in eval mode.
    """
    print("\n== Training feature extractor (for FID + mode coverage) ==")
    extractor = LeNetFeatureExtractor().to(device)
    opt = torch.optim.Adam(extractor.parameters(), lr=1e-3)
    X_01 = (X_real + 1.0) / 2.0  # [0, 1] for classifier

    for epoch in range(epochs):
        losses = []
        for xb, yb in DataLoader(
            TensorDataset(X_01, y_real), batch_size=256, shuffle=True
        ):
            loss = F.cross_entropy(extractor(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            acc = (extractor(X_01[:10000]).argmax(-1) == y_real[:10000]).float().mean()
        print(f"  epoch {epoch+1}/{epochs}  loss={np.mean(losses):.3f}  acc={acc:.3f}")

    extractor.eval()
    return extractor


# ════════════════════════════════════════════════════════════════════════
# FID computation
# ════════════════════════════════════════════════════════════════════════
def compute_fid(
    extractor: LeNetFeatureExtractor,
    real: torch.Tensor,
    generated: torch.Tensor,
) -> float:
    """Frechet Inception Distance between real and generated images.

    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))

    Lower FID = closer to real distribution = better generator.
    Uses eigendecomposition (no scipy dependency).
    """
    extractor.eval()

    def _extract(images: torch.Tensor) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for i in range(0, len(images), 512):
                feats.append(
                    extractor.extract_features(images[i : i + 512]).cpu().numpy()
                )
        return np.concatenate(feats)

    rf, gf = _extract(real), _extract(generated)
    mu_r, mu_g = rf.mean(0), gf.mean(0)
    sig_r, sig_g = np.cov(rf, rowvar=False), np.cov(gf, rowvar=False)

    diff = mu_r - mu_g
    product = sig_r @ sig_g
    eigvals, eigvecs = np.linalg.eigh(product)
    eigvals = np.maximum(eigvals, 0.0)
    sqrt_prod = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T

    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * sqrt_prod))


# ════════════════════════════════════════════════════════════════════════
# Mode coverage diagnostic
# ════════════════════════════════════════════════════════════════════════
def mode_coverage(
    G: Generator,
    classifier: LeNetFeatureExtractor,
    device: torch.device,
    n: int = 5000,
) -> tuple[int, dict[int, int], float]:
    """Measure mode coverage: how many of the 10 digit classes the generator produces.

    Returns:
        n_classes: number of unique classes generated
        per_class_counts: dict mapping class -> count
        shannon_entropy: diversity measure (max = log2(10) = 3.32 for uniform)
    """
    G.eval()
    classifier.eval()
    with torch.no_grad():
        fake_01 = (G(torch.randn(n, LATENT_DIM, device=device)) + 1) / 2
        preds = classifier(fake_01).argmax(-1).cpu().numpy()
    unique, counts = np.unique(preds, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    return int(len(unique)), {int(k): int(v) for k, v in zip(unique, counts)}, entropy


# ════════════════════════════════════════════════════════════════════════
# Visualisation helpers
# ════════════════════════════════════════════════════════════════════════
def plot_image_grid(
    images: torch.Tensor,
    nrow: int = 8,
    ncol: int = 8,
    title: str = "Generated Images",
    save_path: str | None = None,
) -> plt.Figure:
    """Plot an 8x8 grid of generated images.

    Args:
        images: (N, 1, 28, 28) tensor in [-1, 1] range
        nrow, ncol: grid dimensions
        title: figure title
        save_path: optional path to save the figure
    """
    n = min(nrow * ncol, len(images))
    fig, axes = plt.subplots(nrow, ncol, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(nrow * ncol):
        ax = axes[i // ncol][i % ncol]
        if i < n:
            img = images[i].detach().cpu().squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] -> [0,1]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_latent_interpolation(
    G: Generator,
    device: torch.device,
    n_steps: int = 10,
    n_rows: int = 5,
    title: str = "Latent Space Interpolation",
    save_path: str | None = None,
) -> plt.Figure:
    """Interpolate between pairs of random latent vectors.

    Shows smooth transitions between generated images — evidence that
    the generator has learned a continuous manifold, not memorised digits.
    """
    G.eval()
    fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 1.2, n_rows * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    with torch.no_grad():
        for row in range(n_rows):
            z1 = torch.randn(1, LATENT_DIM, device=device)
            z2 = torch.randn(1, LATENT_DIM, device=device)
            for col in range(n_steps):
                alpha = col / (n_steps - 1)
                z = (1 - alpha) * z1 + alpha * z2
                img = G(z).squeeze().cpu().numpy()
                img = (img + 1) / 2
                axes[row][col].imshow(img, cmap="gray", vmin=0, vmax=1)
                axes[row][col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_training_progression(
    G: Generator,
    device: torch.device,
    epoch_snapshots: dict[int, dict],
    title: str = "Training Progression",
    save_path: str | None = None,
) -> plt.Figure:
    """Plot generated images at different training epochs.

    Shows how generation quality improves over training — from random
    noise to recognisable digits.

    Args:
        epoch_snapshots: dict of {epoch: state_dict} captured during training
    """
    n_snapshots = len(epoch_snapshots)
    n_samples = 8
    fig, axes = plt.subplots(
        n_snapshots, n_samples, figsize=(n_samples * 1.4, n_snapshots * 1.6)
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    fixed_z = torch.randn(n_samples, LATENT_DIM, device=device)

    for row, (epoch, state_dict) in enumerate(sorted(epoch_snapshots.items())):
        G.load_state_dict(state_dict)
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_z)
        for col in range(n_samples):
            ax = axes[row][col] if n_snapshots > 1 else axes[col]
            img = imgs[col].squeeze().cpu().numpy()
            img = (img + 1) / 2
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_loss_curves(
    g_losses: list[float],
    d_losses: list[float],
    title: str = "Training Dynamics",
    g_label: str = "Generator",
    d_label: str = "Discriminator",
    save_path: str | None = None,
) -> plt.Figure:
    """Plot G vs D loss curves across epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(g_losses) + 1)

    # Individual losses
    ax1.plot(epochs, g_losses, "b-", linewidth=2, label=g_label)
    ax1.plot(epochs, d_losses, "r-", linewidth=2, label=d_label)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("G vs D Loss", fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # G/D ratio — healthy GAN has ratio near 1
    ratio = [g / (d + 1e-8) for g, d in zip(g_losses, d_losses)]
    ax2.plot(epochs, ratio, "g-", linewidth=2)
    ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Balanced (1.0)")
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("G/D Loss Ratio", fontsize=12)
    ax2.set_title("Training Balance", fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


# ════════════════════════════════════════════════════════════════════════
# Model registration helper
# ════════════════════════════════════════════════════════════════════════
def register_generator(
    registry: ModelRegistry | None,
    name: str,
    model: Generator,
    fid: float,
    coverage: int,
    entropy: float,
) -> "ModelVersion | None":
    """Register a trained generator in ModelRegistry with quality metrics."""
    if registry is None:
        print(f"  ModelRegistry not available — skipping {name}")
        return None

    async def _register():
        ver = await registry.register_model(
            name=f"m5_{name}",
            artifact=pickle.dumps(model.state_dict()),
            metrics=[
                MetricSpec(name="fid_score", value=fid),
                MetricSpec(name="mode_coverage", value=float(coverage)),
                MetricSpec(name="class_entropy", value=entropy),
            ],
        )
        print(
            f"  Registered {name}: v={ver.version}, FID={fid:.2f}, "
            f"coverage={coverage}/10"
        )
        return ver

    return asyncio.run(_register())



# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 5.2: WGAN-GP (Wasserstein GAN with Gradient Penalty)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why vanilla GAN training is unstable (JS divergence saturation)
#   - Wasserstein distance intuition: the "earth mover's distance"
#   - Gradient penalty vs weight clipping — why GP wins
#   - Train a WGAN-GP with critic (not discriminator) architecture
#   - Compare training stability against vanilla GAN
#   - Apply privacy-preserving synthetic medical imaging for a
#     Singapore hospital under PDPA
#
# PREREQUISITES: ex_5/01_vanilla_gan.py (vanilla GAN fundamentals)
# ESTIMATED TIME: ~45 min
# DATASET: MNIST — 60,000 real 28x28 grayscale digits
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

    LATENT_DIM,
    OUTPUT_DIR,
    Generator,
    Discriminator,
    init_environment,
    load_mnist,
    setup_engines,
    close_engines,
    plot_image_grid,
    plot_latent_interpolation,
    plot_training_progression,
    plot_loss_curves,
)



## PHASE 1 — THEORY: Why Vanilla GANs Fail and How Wasserstein Fixes It


THE PROBLEM WITH VANILLA GANS:

  Vanilla GAN uses Jensen-Shannon (JS) divergence to measure how
  different the real and generated distributions are. JS divergence
  has a fatal flaw: when the two distributions don't overlap at all
  (which happens early in training), JS divergence is a CONSTANT.

  A constant means zero gradient. Zero gradient means the generator
  learns nothing. Training stalls or collapses.

  Think of it this way: the detective is SO good that the counterfeiter
  gets no useful feedback — just "everything you make is obviously fake."
  No signal about HOW to improve.

  THE WASSERSTEIN FIX:

  Wasserstein distance (Earth Mover's Distance) measures how much
  "work" it takes to reshape one distribution into another. Imagine
  you have two piles of dirt (real and generated distributions). The
  Wasserstein distance is the minimum total distance you'd need to
  move dirt to transform one pile into the other.

  Key advantage: Wasserstein distance is SMOOTH. Even when distributions
  don't overlap, it still provides meaningful gradients — the generator
  always knows which direction to improve.

  GRADIENT PENALTY (Gulrajani 2017):

  The original WGAN (Arjovsky 2017) enforced the Lipschitz constraint
  by clipping weights to [-c, c]. This caused "capacity underuse":
  most weights cluster at the clip boundaries, wasting the network's
  representational power.

  Gradient penalty is smarter: instead of restricting weights, it
  directly penalises the critic's gradients when they deviate from
  norm 1. This is a soft constraint that preserves the full network
  capacity while enforcing the Lipschitz condition.

  CRITIC VS DISCRIMINATOR:

  In WGAN, the "discriminator" is called a CRITIC because it outputs
  an unbounded real number (the Wasserstein estimate), not a
  probability in [0, 1]. The critic scores images — higher score
  means "more real-looking" — without the sigmoid bottleneck.

  WGAN LOSSES:
    L_critic = E[critic(fake)] - E[critic(real)] + lambda * GP
    L_G      = -E[critic(fake)]
    GP       = E[(||grad critic(interpolated)||_2 - 1)^2]


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: From BCE to Wasserstein Distance")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: Gradient Penalty Implementation


1. Sample alpha ~ U(0, 1) per example
    2. Create interpolated images: x_hat = alpha * real + (1 - alpha) * fake
    3. Compute critic output on interpolated images
    4. Compute gradient of critic w.r.t. interpolated images
    5. Penalise when gradient norm deviates from 1

    The penalty term enforces the 1-Lipschitz constraint on the critic
    without weight clipping, preserving full network capacity.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: WGAN-GP Architecture + Gradient Penalty")
print("=" * 70)

device = init_environment()
X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()


def gradient_penalty(
    D: nn.Module, real: torch.Tensor, fake: torch.Tensor
) -> torch.Tensor:
    batch = real.size(0)
    # TODO: Sample alpha ~ U(0, 1) with shape (batch, 1, 1, 1) for broadcasting
    # Hint: alpha = torch.rand(batch, 1, 1, 1, device=real.device)
    alpha = ____

    # TODO: Create interpolated images between real and fake
    # Hint: interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    interp = ____

    # TODO: Get critic scores on interpolated images
    # Hint: d_interp = D(interp)
    d_interp = ____

    # TODO: Compute gradients of critic output w.r.t. interpolated images
    # Hint: Use torch.autograd.grad with create_graph=True, retain_graph=True
    grad = torch.autograd.grad(
        outputs=d_interp,
        inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    # TODO: Gradient penalty = mean of (||grad||_2 - 1)^2
    # Hint: ((grad.reshape(batch, -1).norm(2, dim=1) - 1) ** 2).mean()
    return ____


# Verify gradient penalty computation
_G_test = Generator().to(device)
_D_test = Discriminator().to(device)
_z = torch.randn(4, LATENT_DIM, device=device)
_real_batch = X_real[:4]
_fake_batch = _G_test(_z)
_gp = gradient_penalty(_D_test, _real_batch, _fake_batch)

print(f"\n  Gradient penalty on test batch: {_gp.item():.4f}")
print("  (should be a positive number — penalises gradient norm != 1)")



### Checkpoint 1


In [ ]:
assert _gp.item() >= 0, "Gradient penalty must be non-negative"
assert _gp.requires_grad, "GP must be differentiable (part of training graph)"
del _G_test, _D_test, _z, _real_batch, _fake_batch, _gp
print("\n--- Checkpoint 1 passed --- gradient penalty verified\n")



## PHASE 3 — TRAIN: WGAN-GP with Critic Training


WGAN-GP training differs from vanilla GAN in three key ways:
  1. Critic trains {N_CRITIC}x more often than generator
     (critic needs to be a good Wasserstein estimator)
  2. No sigmoid — critic outputs unbounded scores
  3. Gradient penalty (lambda={GP_LAMBDA}) replaces weight clipping


Train WGAN-GP with gradient penalty, logging to ExperimentTracker.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: WGAN-GP (5 Critic Steps per Generator Step)")
print("=" * 70)

EPOCHS = 20
LR = 1e-4
N_CRITIC = 5
GP_LAMBDA = 10.0

print(
)


async def train_wgan_gp(
    epochs: int = EPOCHS,
    lr: float = LR,
    n_critic: int = N_CRITIC,
    lam: float = GP_LAMBDA,
):
    G = Generator().to(device)
    D = Discriminator().to(device)
    # NOTE: WGAN-GP uses betas=(0.5, 0.9) not (0.5, 0.999) — lower beta2
    # for more aggressive gradient adaptation (recommended by Gulrajani 2017)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.9))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.9))
    g_losses, d_losses = [], []
    epoch_snapshots = {}

    epoch_snapshots[0] = copy.deepcopy(G.state_dict())

    async with tracker.run(experiment_name=exp_name, run_name="wgan_gp") as ctx:
        await ctx.log_params(
            {
                "architecture": "WGAN_GP_MLP",
                "latent_dim": str(LATENT_DIM),
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": "128",
                "loss_type": "Wasserstein+GP",
                "n_critic": str(n_critic),
                "gp_lambda": str(lam),
                "optimizer": "Adam(0.5,0.9)",
            }
        )

        for epoch in range(epochs):
            eg, ed = [], []
            for (real_batch,) in real_loader:
                bs = real_batch.size(0)

                # ── Train Critic (n_critic steps per G step) ─────────
                for _ in range(n_critic):
                    z = torch.randn(bs, LATENT_DIM, device=device)
                    fake = G(z).detach()
                    gp = gradient_penalty(D, real_batch, fake)
                    # TODO: Wasserstein critic loss + gradient penalty
                    # Critic wants: high scores on real, low scores on fake
                    # loss = E[D(fake)] - E[D(real)] + lambda * GP
                    # Hint: loss_d = D(fake).mean() - D(real_batch).mean() + lam * gp
                    loss_d = ____
                    opt_d.zero_grad()
                    loss_d.backward()
                    opt_d.step()

                # ── Train Generator ──────────────────────────────────
                z = torch.randn(bs, LATENT_DIM, device=device)
                # TODO: G loss — maximise critic score on fakes
                # Hint: loss_g = -D(G(z)).mean()
                loss_g = ____
                opt_g.zero_grad()
                loss_g.backward()
                opt_g.step()

                eg.append(loss_g.item())
                ed.append(loss_d.item())

            avg_g, avg_d = float(np.mean(eg)), float(np.mean(ed))
            g_losses.append(avg_g)
            d_losses.append(avg_d)
            await ctx.log_metrics({"g_loss": avg_g, "d_loss": avg_d}, step=epoch + 1)
            print(
                f"  [WGAN-GP] epoch {epoch+1:2d}/{epochs}  "
                f"critic={avg_d:.3f}  G={avg_g:.3f}"
            )

            if (epoch + 1) in {1, 5, 10, 15, 20}:
                epoch_snapshots[epoch + 1] = copy.deepcopy(G.state_dict())

        await ctx.log_metrics(
            {"final_g_loss": g_losses[-1], "final_d_loss": d_losses[-1]}
        )

    return G, g_losses, d_losses, epoch_snapshots


print("\n  Training WGAN-GP on full MNIST (60K images)...")
G_wgan, wgan_g_losses, wgan_d_losses, wgan_snapshots  = await train_wgan_gp()



### Checkpoint 2


In [ ]:
assert (
    len(wgan_g_losses) == EPOCHS
), f"Expected {EPOCHS} epochs, got {len(wgan_g_losses)}"
# INTERPRETATION: Unlike vanilla GAN's BCE loss, the WGAN critic loss
# approximates the Wasserstein distance — a MEANINGFUL quality metric.
# Lower critic loss = distributions are closer = better generation.
# The loss should decrease smoothly, unlike vanilla GAN's oscillation.
print("\n--- Checkpoint 2 passed --- WGAN-GP trained\n")



## PHASE 4 — VISUALISE: Quality Comparison and Stability Analysis


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: WGAN-GP Quality and Stability")
print("=" * 70)

# 4A: Generated image gallery
print("\n  4A: Gallery of 64 WGAN-GP generated digits")
G_wgan.eval()
with torch.no_grad():
    z_gallery = torch.randn(64, LATENT_DIM, device=device)
    gallery_images = G_wgan(z_gallery)

fig_gallery = plot_image_grid(
    gallery_images,
    title="WGAN-GP — Generated Digits (8x8 Grid)",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_gallery.png"),
)
plt.show()

# 4B: Training progression
print("\n  4B: Training progression — smoother than vanilla GAN")
G_progression = Generator().to(device)
fig_prog = plot_training_progression(
    G_progression,
    device,
    wgan_snapshots,
    title="WGAN-GP — Training Progression",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_progression.png"),
)
plt.show()
G_wgan.load_state_dict(wgan_snapshots[max(wgan_snapshots.keys())])
G_wgan.eval()

# 4C: Critic loss curve — should be smoother than D loss
print("\n  4C: Critic loss dynamics (should decrease smoothly)")
fig_critic = plot_loss_curves(
    wgan_g_losses,
    wgan_d_losses,
    title="WGAN-GP — Training Dynamics",
    g_label="Generator",
    d_label="Critic (Wasserstein estimate)",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_losses.png"),
)
plt.show()

# 4D: Latent interpolation
print("\n  4D: Latent space interpolation")
fig_interp = plot_latent_interpolation(
    G_wgan,
    device,
    title="WGAN-GP — Latent Interpolation",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_interpolation.png"),
)
plt.show()

# 4E: Side-by-side stability comparison
print("\n  4E: Stability comparison — Vanilla GAN vs WGAN-GP")
fig_compare, axes = plt.subplots(1, 2, figsize=(16, 5))
fig_compare.suptitle(
    "Training Stability: Vanilla GAN vs WGAN-GP",
    fontsize=14,
    fontweight="bold",
)

# TODO: Train a brief vanilla GAN for comparison
# Hint: Reuse Generator/Discriminator, BCEWithLogitsLoss, Adam(0.5, 0.999)
#       Train for min(EPOCHS, 15) epochs, collect G and D losses
print("  Training a brief vanilla GAN for comparison...")
_G_cmp = Generator().to(device)
_D_cmp = Discriminator().to(device)
_opt_g = torch.optim.Adam(_G_cmp.parameters(), lr=2e-4, betas=(0.5, 0.999))
_opt_d = torch.optim.Adam(_D_cmp.parameters(), lr=2e-4, betas=(0.5, 0.999))
_bce = nn.BCEWithLogitsLoss()
_cmp_g, _cmp_d = [], []
for _ep in range(min(EPOCHS, 15)):
    _eg, _ed = [], []
    for (_rb,) in real_loader:
        _bs = _rb.size(0)
        _z = torch.randn(_bs, LATENT_DIM, device=device)
        _fk = _G_cmp(_z).detach()
        # TODO: Vanilla D loss (BCE on real + fake)
        # Hint: _ld = _bce(_D_cmp(_rb), torch.ones(_bs, 1, device=device))
        #            + _bce(_D_cmp(_fk), torch.zeros(_bs, 1, device=device))
        _ld = ____
        _opt_d.zero_grad()
        _ld.backward()
        _opt_d.step()
        _z = torch.randn(_bs, LATENT_DIM, device=device)
        # TODO: Vanilla G loss (fool D)
        # Hint: _lg = _bce(_D_cmp(_G_cmp(_z)), torch.ones(_bs, 1, device=device))
        _lg = ____
        _opt_g.zero_grad()
        _lg.backward()
        _opt_g.step()
        _eg.append(_lg.item())
        _ed.append(_ld.item())
    _cmp_g.append(float(np.mean(_eg)))
    _cmp_d.append(float(np.mean(_ed)))
del _G_cmp, _D_cmp, _opt_g, _opt_d

# Vanilla GAN D loss (oscillating)
van_epochs = range(1, len(_cmp_d) + 1)
axes[0].plot(van_epochs, _cmp_d, "r-", linewidth=2, alpha=0.8, label="Vanilla D Loss")
axes[0].plot(van_epochs, _cmp_g, "b-", linewidth=2, alpha=0.8, label="Vanilla G Loss")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss (BCE)", fontsize=12)
axes[0].set_title("Vanilla GAN: Oscillating Losses", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# WGAN-GP critic loss (smooth descent)
wgan_epochs = range(1, len(wgan_d_losses) + 1)
axes[1].plot(
    wgan_epochs, wgan_d_losses, "r-", linewidth=2, alpha=0.8, label="Critic Loss"
)
axes[1].plot(wgan_epochs, wgan_g_losses, "b-", linewidth=2, alpha=0.8, label="G Loss")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss (Wasserstein)", fontsize=12)
axes[1].set_title("WGAN-GP: Smooth Convergence", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_compare.savefig(
    str(OUTPUT_DIR / "ex_5_02_stability_comparison.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Stability comparison saved")



### Checkpoint 3


In [ ]:
import os

assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_02_wgan_gallery.png")
), "WGAN gallery should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_02_stability_comparison.png")
), "Stability comparison should exist"
print("\n--- Checkpoint 3 passed --- WGAN-GP visualisations complete\n")

# Interactive training curves with ModelVisualizer
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()
fig_html = viz.training_history(
    metrics={
        "WGAN-GP G loss": wgan_g_losses,
        "WGAN-GP Critic loss": wgan_d_losses,
    },
    x_label="Epoch",
    y_label="Loss",
)
fig_html.write_html(str(OUTPUT_DIR / "ex_5_02_wgan_training.html"))
print("  Interactive training curves saved to ex_5_02_wgan_training.html")



## PHASE 5 — APPLY: Privacy-Preserving Synthetic Medical Images at NUH


BUSINESS SCENARIO:
  You are an AI researcher at the National University Hospital (NUH)
  in Singapore. Your team is developing an automated chest X-ray
  screening model for tuberculosis (TB). Training requires thousands
  of annotated X-ray images, but sharing real patient scans outside
  the hospital is prohibited by PDPA and the Ministry of Health (MOH)
  data governance framework.

  Other hospitals want to train their own TB screening models but
  lack sufficient local data. NUH cannot share real patient images,
  but CAN share a trained GAN that generates synthetic X-ray-like
  images preserving the visual patterns of TB without containing
  any real patient data.

  SOLUTION: Train a WGAN-GP on NUH's X-ray dataset (kept internal).
  Share only the trained generator. Partner hospitals generate synthetic
  training data locally — no real patient data ever leaves NUH.

  WHY WGAN-GP (not vanilla GAN):
  Medical images demand training STABILITY. A mode-collapsed GAN
  that only generates one type of X-ray would train a biased screening
  model. WGAN-GP's smooth Wasserstein gradients prevent collapse,
  ensuring the synthetic dataset covers the full range of TB
  presentations (mild, moderate, severe, bilateral, unilateral).

  BUSINESS IMPACT:
  - 3 partner hospitals gain access to TB screening AI
  - Zero real patient data shared (PDPA + MOH compliant)
  - Screening model sensitivity: 89% (synthetic) vs 92% (real data)
  - Cost savings: S$1.2M/year across 3 hospitals (reduced manual reads)
  - Time to diagnosis: 48 hours → 4 hours for preliminary screen


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Medical Images for NUH")
print("=" * 70)
print(
)

# Simulate the medical imaging scenario with MNIST as proxy
# (real deployment would use chest X-ray datasets like CheXpert)
print("\n  Simulating the NUH medical imaging scenario...")

# Step 1: Generate a large synthetic dataset from the trained WGAN-GP
G_wgan.eval()
n_synthetic = 10000
with torch.no_grad():
    # TODO: Generate synthetic medical images from latent space
    # Hint: z_medical = torch.randn(n_synthetic, LATENT_DIM, device=device)
    #       X_medical_synthetic = G_wgan(z_medical)
    z_medical = ____
    X_medical_synthetic = ____

print(f"  Synthetic 'X-ray' images generated: {n_synthetic}")

# Step 2: Compare quality — real vs WGAN-GP synthetic vs vanilla GAN
# Train a quick vanilla GAN to show the quality difference
print("  Training vanilla GAN for quality comparison...")
_G_van = Generator().to(device)
_D_van = Discriminator().to(device)
_opt_gv = torch.optim.Adam(_G_van.parameters(), lr=2e-4, betas=(0.5, 0.999))
_opt_dv = torch.optim.Adam(_D_van.parameters(), lr=2e-4, betas=(0.5, 0.999))
_bce_van = nn.BCEWithLogitsLoss()
for _ep in range(10):
    for (_rb,) in real_loader:
        _bs = _rb.size(0)
        _z = torch.randn(_bs, LATENT_DIM, device=device)
        _fk = _G_van(_z).detach()
        _ld = _bce_van(_D_van(_rb), torch.ones(_bs, 1, device=device)) + _bce_van(
            _D_van(_fk), torch.zeros(_bs, 1, device=device)
        )
        _opt_dv.zero_grad()
        _ld.backward()
        _opt_dv.step()
        _z = torch.randn(_bs, LATENT_DIM, device=device)
        _lg = _bce_van(_D_van(_G_van(_z)), torch.ones(_bs, 1, device=device))
        _opt_gv.zero_grad()
        _lg.backward()
        _opt_gv.step()

_G_van.eval()
with torch.no_grad():
    X_vanilla_synthetic = _G_van(torch.randn(64, LATENT_DIM, device=device))
del _D_van, _opt_gv, _opt_dv

# Step 3: Visual comparison — Real vs Vanilla GAN vs WGAN-GP
fig_med, axes = plt.subplots(3, 8, figsize=(16, 7))
fig_med.suptitle(
    "Medical Image Quality Comparison\n"
    "Real Patient Scans vs Vanilla GAN vs WGAN-GP Synthetic",
    fontsize=14,
    fontweight="bold",
)

rng = np.random.default_rng(42)
real_sample_idx = rng.choice(len(X_real), 8, replace=False)

for col in range(8):
    # Row 1: Real images
    img = X_real[real_sample_idx[col]].squeeze().cpu().numpy()
    axes[0][col].imshow((img + 1) / 2, cmap="gray", vmin=0, vmax=1)
    axes[0][col].axis("off")
    if col == 0:
        axes[0][col].set_ylabel("Real", fontsize=12, rotation=0, labelpad=50)

    # Row 2: Vanilla GAN
    img = X_vanilla_synthetic[col].squeeze().cpu().numpy()
    axes[1][col].imshow((img + 1) / 2, cmap="gray", vmin=0, vmax=1)
    axes[1][col].axis("off")
    if col == 0:
        axes[1][col].set_ylabel("Vanilla", fontsize=12, rotation=0, labelpad=50)

    # Row 3: WGAN-GP
    img = X_medical_synthetic[col].squeeze().cpu().numpy()
    axes[2][col].imshow((img + 1) / 2, cmap="gray", vmin=0, vmax=1)
    axes[2][col].axis("off")
    if col == 0:
        axes[2][col].set_ylabel("WGAN-GP", fontsize=12, rotation=0, labelpad=50)

plt.tight_layout()
fig_med.savefig(
    str(OUTPUT_DIR / "ex_5_02_medical_comparison.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Medical image comparison saved")
del _G_van, X_vanilla_synthetic

# Step 4: Diagnostic model comparison — trained on real vs synthetic
print("\n  Training diagnostic models: real data vs synthetic data...")


# Simple classifier to simulate the diagnostic model
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


# TODO: Train Model A on real data, Model B on synthetic data, compare accuracy
# Model A: trained on real data (5000 samples, 3 epochs)
model_real = SimpleClassifier().to(device)
opt_real = torch.optim.Adam(model_real.parameters(), lr=1e-3)
X_01 = (X_real + 1.0) / 2.0
for _ in range(3):
    for xb, yb in torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_01[:5000], y_real[:5000]),
        batch_size=256,
        shuffle=True,
    ):
        loss = torch.nn.functional.cross_entropy(model_real(xb), yb)
        opt_real.zero_grad()
        loss.backward()
        opt_real.step()

# Model B: trained on synthetic data (use generated images with pseudo-labels)
model_synth = SimpleClassifier().to(device)
opt_synth = torch.optim.Adam(model_synth.parameters(), lr=1e-3)

# TODO: Generate pseudo-labels for synthetic images using the real-data model
# Hint: synth_01 = (X_medical_synthetic + 1) / 2
#       pseudo_labels = model_real(synth_01).argmax(-1)
with torch.no_grad():
    synth_01 = ____
    pseudo_labels = ____

for _ in range(3):
    for xb, yb in torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(synth_01, pseudo_labels),
        batch_size=256,
        shuffle=True,
    ):
        loss = torch.nn.functional.cross_entropy(model_synth(xb), yb)
        opt_synth.zero_grad()
        loss.backward()
        opt_synth.step()

# Evaluate both on held-out real test data
X_test_01 = X_01[50000:]
y_test = y_real[50000:]
with torch.no_grad():
    acc_real = (model_real(X_test_01).argmax(-1) == y_test).float().mean().item()
    acc_synth = (model_synth(X_test_01).argmax(-1) == y_test).float().mean().item()

print(f"\n  Diagnostic model accuracy (trained on real data):      {acc_real:.1%}")
print(f"  Diagnostic model accuracy (trained on synthetic data): {acc_synth:.1%}")
print(f"  Performance gap: {abs(acc_real - acc_synth):.1%}")

# Step 5: Stakeholder-ready summary
print("\n  ┌────────────────────────────────────────────────────────────┐")
print("  │  STAKEHOLDER SUMMARY: NUH Synthetic Medical Imaging       │")
print("  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Synthetic images generated:   {n_synthetic:>8,}                  │")
print(f"  │  Real-data model accuracy:     {acc_real:>8.1%}                  │")
print(f"  │  Synthetic-data model accuracy: {acc_synth:>7.1%}                  │")
print(
    f"  │  Performance gap:              {abs(acc_real - acc_synth):>8.1%}                  │"
)
print("  │  Patient data shared:          ZERO                       │")
print("  │  PDPA compliance:              FULL                       │")
print("  │  Partner hospitals enabled:    3                          │")
print("  │  Annual cost savings:          S$1.2M (3 hospitals)       │")
print("  │  Time to preliminary screen:   48h → 4h                   │")
print("  └────────────────────────────────────────────────────────────┘")



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_02_medical_comparison.png")
), "Medical comparison should exist"
assert acc_synth > 0.3, "Synthetic-trained model should be non-trivial"
print("\n--- Checkpoint 4 passed --- NUH medical application complete\n")



## Cleanup


In [ ]:
await close_engines(conn)



## REFLECTION


WGAN-GP THEORY AND PRACTICE:
  [x] Why vanilla GAN fails: JS divergence gives zero gradient when
      distributions don't overlap (early training = no learning signal)
  [x] Wasserstein distance: "earth mover's distance" — smooth gradients
      that always point toward improvement
  [x] Gradient penalty replaces weight clipping (Gulrajani 2017):
      soft constraint preserves full network capacity
  [x] Critic (not discriminator): unbounded score, no sigmoid bottleneck
  [x] 5 critic steps per generator step for accurate Wasserstein estimation

  VISUAL INTUITION:
  [x] WGAN-GP gallery vs vanilla GAN — sharper, more diverse digits
  [x] Critic loss decreases smoothly (unlike vanilla GAN oscillation)
  [x] Side-by-side stability comparison proves WGAN-GP advantage
  [x] Latent interpolation shows continuous learned manifold

  REAL-WORLD APPLICATION:
  [x] Privacy-preserving synthetic medical images for NUH
  [x] PDPA compliance: trained generator shared, real data stays internal
  [x] Diagnostic model trained on synthetic data achieves comparable accuracy
  [x] Business impact: 3 partner hospitals, S$1.2M annual savings

  KEY INSIGHT — WHEN TO USE WGAN-GP:
  Use WGAN-GP when training stability matters (medical, financial,
  safety-critical). Vanilla GAN is simpler but unreliable. WGAN-GP's
  smooth gradients make training predictable and debuggable.

  Next: Exercise 5.3 — GAN Evaluation and Model Registry.
  How do you PROVE synthetic data is good enough for production?
  FID scores, mode coverage analysis, and quality assurance pipelines.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — WGAN-GP (Wasserstein distance = smooth gradient)


In [ ]:
# The core WGAN-GP selling point is that the critic provides a
# GRADIENT EVERYWHERE — unlike vanilla GAN where the saturating BCE
# kills Generator gradients when D wins. We expect the Blood Test
# to read healthier than 01_vanilla_gan: no vanishing, no mode
# collapse signature in the activations. The Stethoscope should
# show critic loss decreasing smoothly as the Wasserstein distance
# shrinks — NOT the oscillating adversarial dance of vanilla GAN.
import torch.nn.functional as _F


def _g_loss(m, batch):
    bs = batch[0].size(0)
    z = torch.randn(bs, LATENT_DIM, device=device)
    fake = m(z)
    return _F.mse_loss(fake, batch[0])


print("\n── Diagnostic Report (WGAN-GP Generator) ──")
g_diag, g_findings = run_diagnostic_checkpoint(
    G_wgan,
    real_loader,
    _g_loss,
    title="WGAN-GP — Generator",
    n_batches=6,
    train_losses=wgan_g_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad (WGAN-GP Generator)


In [ ]:
#   [✓] Gradient flow (HEALTHY): min RMS ~2.8e-04 across G layers.
#       Contrast 01's vanilla G at 5e-6 under D dominance.
#       The Wasserstein objective is unsaturating — gradient
#       survives even when critic perfectly separates real/fake.
#   [✓] Activations   (HEALTHY): tanh output entropy across the
#       batch is high — digits of varied shape, no mode collapse.
#       Visible signature of diverse latent -> diverse output.
#   [✓] Loss trend    (HEALTHY): critic (Wasserstein) loss
#       descends smoothly with slope -4.2e-03/epoch — this is the
#       MEANINGFUL quality metric that vanilla GAN BCE cannot give.



## Final G loss ~0.18, critic loss ~-2.3 (trend: monotonically toward 0).

STUDENT INTERPRETATION GUIDE — reading the WGAN-GP Prescription Pad:

 [BLOOD TEST — GRADIENT EVERYWHERE] G's gradient RMS holds
    steady ~1e-4 across training. Compare vanilla GAN (ex_5/01)
    where the same instrument reads WARNING (RMS < 1e-5) the
    moment D_loss approached 0. The gradient penalty is the
    mechanism: it enforces 1-Lipschitz on the critic, which
    bounds the gradient magnitude but also PREVENTS it from
    vanishing. Slide 5.5 covers this — the Earth-Mover distance
    is continuous everywhere, unlike JS divergence.
    >> Prescription: if G RMS still drops below 1e-5 here,
       the gradient penalty coefficient (lambda_gp=10) is too
       low or the critic is under-trained (increase n_critic).

 [X-RAY — NO MODE COLLAPSE] Activation entropy across the batch
    confirms visual diversity. The gallery (below) should show
    varied digits, not 50 copies of a "7". WGAN-GP's Lipschitz
    constraint discourages the critic from sharp rejection of
    minority modes, so G is not punished for diversity.
    >> Prescription: persistent mode collapse even with WGAN-GP
       means the critic architecture is too shallow OR the
       gradient penalty is mis-implemented (check sample points
       are interpolated between real and fake, not just on real).

 [STETHOSCOPE — MEANINGFUL QUALITY METRIC] Critic loss ≈ negative
    Wasserstein distance. It DECREASES as generation quality
    improves — unlike BCE which oscillates. You can actually
    monitor this for early stopping, something impossible with
    vanilla GAN. This is a textbook plot: smooth descent, tight
    error bars, no mode-collapse cliff.
    >> Prescription: if critic loss oscillates like vanilla GAN,
       check that gradient penalty is actually being applied
       (common bug: lambda_gp=0 silent default).

 FIVE-INSTRUMENT TAKEAWAY: WGAN-GP inverts every red flag of
 vanilla GAN — gradient healthy, activations diverse, loss
 meaningful. This is the architectural fix that unlocks
 training stability. Move to ex_5/03 to quantify quality with
 FID, because even WGAN-GP's loss doesn't answer "how real?"


## PHASE 4 — VISUALISE: Quality Comparison and Stability Analysis


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: WGAN-GP Quality and Stability")
print("=" * 70)

# 4A: Generated image gallery
print("\n  4A: Gallery of 64 WGAN-GP generated digits")
G_wgan.eval()
with torch.no_grad():
    z_gallery = torch.randn(64, LATENT_DIM, device=device)
    gallery_images = G_wgan(z_gallery)

fig_gallery = plot_image_grid(
    gallery_images,
    title="WGAN-GP — Generated Digits (8x8 Grid)",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_gallery.png"),
)
plt.show()

# 4B: Training progression
print("\n  4B: Training progression — smoother than vanilla GAN")
G_progression = Generator().to(device)
fig_prog = plot_training_progression(
    G_progression,
    device,
    wgan_snapshots,
    title="WGAN-GP — Training Progression",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_progression.png"),
)
plt.show()
G_wgan.load_state_dict(wgan_snapshots[max(wgan_snapshots.keys())])
G_wgan.eval()

# 4C: Critic loss curve — should be smoother than D loss
print("\n  4C: Critic loss dynamics (should decrease smoothly)")
fig_critic = plot_loss_curves(
    wgan_g_losses,
    wgan_d_losses,
    title="WGAN-GP — Training Dynamics",
    g_label="Generator",
    d_label="Critic (Wasserstein estimate)",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_losses.png"),
)
plt.show()

# 4D: Latent interpolation
print("\n  4D: Latent space interpolation")
fig_interp = plot_latent_interpolation(
    G_wgan,
    device,
    title="WGAN-GP — Latent Interpolation",
    save_path=str(OUTPUT_DIR / "ex_5_02_wgan_interpolation.png"),
)
plt.show()

# 4E: Side-by-side stability comparison
print("\n  4E: Stability comparison — Vanilla GAN vs WGAN-GP")
fig_compare, axes = plt.subplots(1, 2, figsize=(16, 5))
fig_compare.suptitle(
    "Training Stability: Vanilla GAN vs WGAN-GP",
    fontsize=14,
    fontweight="bold",
)

# Load vanilla GAN losses from previous run if available, otherwise note
# (In practice both files run in sequence; we re-train a small vanilla GAN
# for comparison if the previous data isn't available)
try:
    # Import from sibling module — but we'll just retrain briefly for comparison
    raise ImportError("Using local comparison")
except ImportError:
    # Quick vanilla GAN training for comparison plot
    print("  Training a brief vanilla GAN for comparison...")
    _G_cmp = Generator().to(device)
    _D_cmp = Discriminator().to(device)
    _opt_g = torch.optim.Adam(_G_cmp.parameters(), lr=2e-4, betas=(0.5, 0.999))
    _opt_d = torch.optim.Adam(_D_cmp.parameters(), lr=2e-4, betas=(0.5, 0.999))
    _bce = nn.BCEWithLogitsLoss()
    _cmp_g, _cmp_d = [], []
    for _ep in range(min(EPOCHS, 15)):
        _eg, _ed = [], []
        for (_rb,) in real_loader:
            _bs = _rb.size(0)
            _z = torch.randn(_bs, LATENT_DIM, device=device)
            _fk = _G_cmp(_z).detach()
            _ld = _bce(_D_cmp(_rb), torch.ones(_bs, 1, device=device)) + _bce(
                _D_cmp(_fk), torch.zeros(_bs, 1, device=device)
            )
            _opt_d.zero_grad()
            _ld.backward()
            _opt_d.step()
            _z = torch.randn(_bs, LATENT_DIM, device=device)
            _lg = _bce(_D_cmp(_G_cmp(_z)), torch.ones(_bs, 1, device=device))
            _opt_g.zero_grad()
            _lg.backward()
            _opt_g.step()
            _eg.append(_lg.item())
            _ed.append(_ld.item())
        _cmp_g.append(float(np.mean(_eg)))
        _cmp_d.append(float(np.mean(_ed)))
    del _G_cmp, _D_cmp, _opt_g, _opt_d

# Vanilla GAN D loss (oscillating)
van_epochs = range(1, len(_cmp_d) + 1)
axes[0].plot(van_epochs, _cmp_d, "r-", linewidth=2, alpha=0.8, label="Vanilla D Loss")
axes[0].plot(van_epochs, _cmp_g, "b-", linewidth=2, alpha=0.8, label="Vanilla G Loss")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss (BCE)", fontsize=12)
axes[0].set_title("Vanilla GAN: Oscillating Losses", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# WGAN-GP critic loss (smooth descent)
wgan_epochs = range(1, len(wgan_d_losses) + 1)
axes[1].plot(
    wgan_epochs, wgan_d_losses, "r-", linewidth=2, alpha=0.8, label="Critic Loss"
)
axes[1].plot(wgan_epochs, wgan_g_losses, "b-", linewidth=2, alpha=0.8, label="G Loss")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss (Wasserstein)", fontsize=12)
axes[1].set_title("WGAN-GP: Smooth Convergence", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_compare.savefig(
    str(OUTPUT_DIR / "ex_5_02_stability_comparison.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Stability comparison saved")



### Checkpoint 3


In [ ]:
import os

assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_02_wgan_gallery.png")
), "WGAN gallery should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_02_stability_comparison.png")
), "Stability comparison should exist"
print("\n--- Checkpoint 3 passed --- WGAN-GP visualisations complete\n")

# Interactive training curves with ModelVisualizer
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()
fig_html = viz.training_history(
    metrics={
        "WGAN-GP G loss": wgan_g_losses,
        "WGAN-GP Critic loss": wgan_d_losses,
    },
    x_label="Epoch",
    y_label="Loss",
)
fig_html.write_html(str(OUTPUT_DIR / "ex_5_02_wgan_training.html"))
print("  Interactive training curves saved to ex_5_02_wgan_training.html")



## PHASE 5 — APPLY: Privacy-Preserving Synthetic Medical Images at NUH


BUSINESS SCENARIO:
  You are an AI researcher at the National University Hospital (NUH)
  in Singapore. Your team is developing an automated chest X-ray
  screening model for tuberculosis (TB). Training requires thousands
  of annotated X-ray images, but sharing real patient scans outside
  the hospital is prohibited by PDPA and the Ministry of Health (MOH)
  data governance framework.

  Other hospitals want to train their own TB screening models but
  lack sufficient local data. NUH cannot share real patient images,
  but CAN share a trained GAN that generates synthetic X-ray-like
  images preserving the visual patterns of TB without containing
  any real patient data.

  SOLUTION: Train a WGAN-GP on NUH's X-ray dataset (kept internal).
  Share only the trained generator. Partner hospitals generate synthetic
  training data locally — no real patient data ever leaves NUH.

  WHY WGAN-GP (not vanilla GAN):
  Medical images demand training STABILITY. A mode-collapsed GAN
  that only generates one type of X-ray would train a biased screening
  model. WGAN-GP's smooth Wasserstein gradients prevent collapse,
  ensuring the synthetic dataset covers the full range of TB
  presentations (mild, moderate, severe, bilateral, unilateral).

  BUSINESS IMPACT:
  - 3 partner hospitals gain access to TB screening AI
  - Zero real patient data shared (PDPA + MOH compliant)
  - Screening model sensitivity: 89% (synthetic) vs 92% (real data)
  - Cost savings: S$1.2M/year across 3 hospitals (reduced manual reads)
  - Time to diagnosis: 48 hours → 4 hours for preliminary screen


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Medical Images for NUH")
print("=" * 70)
print(
)

# Simulate the medical imaging scenario with MNIST as proxy
# (real deployment would use chest X-ray datasets like CheXpert)
print("\n  Simulating the NUH medical imaging scenario...")

# Step 1: Generate a large synthetic dataset from the trained WGAN-GP
G_wgan.eval()
n_synthetic = 10000
with torch.no_grad():
    z_medical = torch.randn(n_synthetic, LATENT_DIM, device=device)
    X_medical_synthetic = G_wgan(z_medical)

print(f"  Synthetic 'X-ray' images generated: {n_synthetic}")

# Step 2: Compare quality — real vs WGAN-GP synthetic vs vanilla GAN
# Train a quick vanilla GAN to show the quality difference
print("  Training vanilla GAN for quality comparison...")
_G_van = Generator().to(device)
_D_van = Discriminator().to(device)
_opt_gv = torch.optim.Adam(_G_van.parameters(), lr=2e-4, betas=(0.5, 0.999))
_opt_dv = torch.optim.Adam(_D_van.parameters(), lr=2e-4, betas=(0.5, 0.999))
_bce_van = nn.BCEWithLogitsLoss()
for _ep in range(10):
    for (_rb,) in real_loader:
        _bs = _rb.size(0)
        _z = torch.randn(_bs, LATENT_DIM, device=device)
        _fk = _G_van(_z).detach()
        _ld = _bce_van(_D_van(_rb), torch.ones(_bs, 1, device=device)) + _bce_van(
            _D_van(_fk), torch.zeros(_bs, 1, device=device)
        )
        _opt_dv.zero_grad()
        _ld.backward()
        _opt_dv.step()
        _z = torch.randn(_bs, LATENT_DIM, device=device)
        _lg = _bce_van(_D_van(_G_van(_z)), torch.ones(_bs, 1, device=device))
        _opt_gv.zero_grad()
        _lg.backward()
        _opt_gv.step()

_G_van.eval()
with torch.no_grad():
    X_vanilla_synthetic = _G_van(torch.randn(64, LATENT_DIM, device=device))
del _D_van, _opt_gv, _opt_dv

# Step 3: Visual comparison — Real vs Vanilla GAN vs WGAN-GP
fig_med, axes = plt.subplots(3, 8, figsize=(16, 7))
fig_med.suptitle(
    "Medical Image Quality Comparison\n"
    "Real Patient Scans vs Vanilla GAN vs WGAN-GP Synthetic",
    fontsize=14,
    fontweight="bold",
)

rng = np.random.default_rng(42)
real_sample_idx = rng.choice(len(X_real), 8, replace=False)

for col in range(8):
    # Row 1: Real images
    img = X_real[real_sample_idx[col]].squeeze().cpu().numpy()
    axes[0][col].imshow((img + 1) / 2, cmap="gray", vmin=0, vmax=1)
    axes[0][col].axis("off")
    if col == 0:
        axes[0][col].set_ylabel("Real", fontsize=12, rotation=0, labelpad=50)

    # Row 2: Vanilla GAN
    img = X_vanilla_synthetic[col].squeeze().cpu().numpy()
    axes[1][col].imshow((img + 1) / 2, cmap="gray", vmin=0, vmax=1)
    axes[1][col].axis("off")
    if col == 0:
        axes[1][col].set_ylabel("Vanilla", fontsize=12, rotation=0, labelpad=50)

    # Row 3: WGAN-GP
    img = X_medical_synthetic[col].squeeze().cpu().numpy()
    axes[2][col].imshow((img + 1) / 2, cmap="gray", vmin=0, vmax=1)
    axes[2][col].axis("off")
    if col == 0:
        axes[2][col].set_ylabel("WGAN-GP", fontsize=12, rotation=0, labelpad=50)

plt.tight_layout()
fig_med.savefig(
    str(OUTPUT_DIR / "ex_5_02_medical_comparison.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Medical image comparison saved")
del _G_van, X_vanilla_synthetic

# Step 4: Diagnostic model comparison — trained on real vs synthetic
print("\n  Training diagnostic models: real data vs synthetic data...")


# Simple classifier to simulate the diagnostic model
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


# Model A: trained on real data
model_real = SimpleClassifier().to(device)
opt_real = torch.optim.Adam(model_real.parameters(), lr=1e-3)
X_01 = (X_real + 1.0) / 2.0
for _ in range(3):
    for xb, yb in torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_01[:5000], y_real[:5000]),
        batch_size=256,
        shuffle=True,
    ):
        loss = torch.nn.functional.cross_entropy(model_real(xb), yb)
        opt_real.zero_grad()
        loss.backward()
        opt_real.step()

# Model B: trained on synthetic data (use generated images with pseudo-labels)
# In a real scenario, a radiologist labels a small synthetic subset
model_synth = SimpleClassifier().to(device)
opt_synth = torch.optim.Adam(model_synth.parameters(), lr=1e-3)

# Generate pseudo-labels using the real-data model
with torch.no_grad():
    synth_01 = (X_medical_synthetic + 1) / 2
    pseudo_labels = model_real(synth_01).argmax(-1)

for _ in range(3):
    for xb, yb in torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(synth_01, pseudo_labels),
        batch_size=256,
        shuffle=True,
    ):
        loss = torch.nn.functional.cross_entropy(model_synth(xb), yb)
        opt_synth.zero_grad()
        loss.backward()
        opt_synth.step()

# Evaluate both on held-out real test data
X_test_01 = X_01[50000:]
y_test = y_real[50000:]
with torch.no_grad():
    acc_real = (model_real(X_test_01).argmax(-1) == y_test).float().mean().item()
    acc_synth = (model_synth(X_test_01).argmax(-1) == y_test).float().mean().item()

print(f"\n  Diagnostic model accuracy (trained on real data):      {acc_real:.1%}")
print(f"  Diagnostic model accuracy (trained on synthetic data): {acc_synth:.1%}")
print(f"  Performance gap: {abs(acc_real - acc_synth):.1%}")

# Step 5: Stakeholder-ready summary
print("\n  ┌────────────────────────────────────────────────────────────┐")
print("  │  STAKEHOLDER SUMMARY: NUH Synthetic Medical Imaging       │")
print("  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Synthetic images generated:   {n_synthetic:>8,}                  │")
print(f"  │  Real-data model accuracy:     {acc_real:>8.1%}                  │")
print(f"  │  Synthetic-data model accuracy: {acc_synth:>7.1%}                  │")
print(
    f"  │  Performance gap:              {abs(acc_real - acc_synth):>8.1%}                  │"
)
print("  │  Patient data shared:          ZERO                       │")
print("  │  PDPA compliance:              FULL                       │")
print("  │  Partner hospitals enabled:    3                          │")
print("  │  Annual cost savings:          S$1.2M (3 hospitals)       │")
print("  │  Time to preliminary screen:   48h → 4h                   │")
print("  └────────────────────────────────────────────────────────────┘")



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_02_medical_comparison.png")
), "Medical comparison should exist"
assert acc_synth > 0.3, "Synthetic-trained model should be non-trivial"
print("\n--- Checkpoint 4 passed --- NUH medical application complete\n")



## Cleanup


In [ ]:
await close_engines(conn)



## REFLECTION


WGAN-GP THEORY AND PRACTICE:
  [x] Why vanilla GAN fails: JS divergence gives zero gradient when
      distributions don't overlap (early training = no learning signal)
  [x] Wasserstein distance: "earth mover's distance" — smooth gradients
      that always point toward improvement
  [x] Gradient penalty replaces weight clipping (Gulrajani 2017):
      soft constraint preserves full network capacity
  [x] Critic (not discriminator): unbounded score, no sigmoid bottleneck
  [x] 5 critic steps per generator step for accurate Wasserstein estimation

  VISUAL INTUITION:
  [x] WGAN-GP gallery vs vanilla GAN — sharper, more diverse digits
  [x] Critic loss decreases smoothly (unlike vanilla GAN oscillation)
  [x] Side-by-side stability comparison proves WGAN-GP advantage
  [x] Latent interpolation shows continuous learned manifold

  REAL-WORLD APPLICATION:
  [x] Privacy-preserving synthetic medical images for NUH
  [x] PDPA compliance: trained generator shared, real data stays internal
  [x] Diagnostic model trained on synthetic data achieves comparable accuracy
  [x] Business impact: 3 partner hospitals, S$1.2M annual savings

  KEY INSIGHT — WHEN TO USE WGAN-GP:
  Use WGAN-GP when training stability matters (medical, financial,
  safety-critical). Vanilla GAN is simpler but unreliable. WGAN-GP's
  smooth gradients make training predictable and debuggable.

  Next: Exercise 5.3 — GAN Evaluation and Model Registry.
  How do you PROVE synthetic data is good enough for production?
  FID scores, mode coverage analysis, and quality assurance pipelines.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

